# Chạy thử DOCX — Pandoc + DeepSeek-OCR

**Chỉ dùng notebook này. Không cần copy thêm cell, clone GitHub hay sửa đường dẫn.**

1. Giữ notebook **Private**, bật **Internet** và chọn **GPU** trong Settings.
2. Bấm **Run All** một lần. Notebook chỉ mở giao diện, chưa chạy OCR.
3. Bấm **Chọn DOCX** ở giao diện bên dưới → đợi hiện tên file → bấm **Chạy OCR**.
4. Đợi hoàn tất rồi bấm **Tải kết quả ZIP**.

Khi chạy, nút đổi chữ thành **Đang chạy — xem trạng thái**. Khung trạng thái ghi bước hiện tại, thời gian và PID; mỗi 5 giây có tín hiệu tiến trình còn chạy kể cả khi chưa có log mới. Đây không phải phần trăm hoàn thành hay bằng chứng tiến trình không bị kẹt. Cài LibreOffice hiện log trực tiếp, có giới hạn chờ. Hoàn tất, lỗi hoặc bị ngắt đều được ghi bằng chữ; không cần đoán theo màu nút.

Kết quả có **de-thi.html** để in ở kích thước ảnh/công thức của DOCX, không dùng kích thước PNG đã phóng lớn cho OCR. Mở HTML rồi bấm **In / lưu PDF**, chọn tỉ lệ 100% và tắt header/footer của trình duyệt. Phần OCR được tách sang bản đối chiếu, không chèn thêm vào trang đề khi in. Nếu LibreOffice Writer chuyển được DOCX, ZIP còn có **de-thi-goc.pdf** giữ bố cục Word gốc (không phải PDF do OCR sinh). HTML có thể khác ngắt trang; PDF có thể thay phông nếu thiếu phông Word.

Nếu xử lý thất bại, giao diện hiển thị lỗi gốc và nút **Tải log lỗi (run.log)**. Gửi log đó để xác định bước/thư viện lỗi; không cần sửa cell.

Nếu DOCX đã nằm trong dataset, mở **Hoặc chọn DOCX đã có trên Kaggle**, bấm **Làm mới** rồi chọn file. Nếu chỉ có một DOCX, notebook tự chọn. Thiếu file/GPU thì nút chạy bị khóa, không cần sửa code.

Lần đầu cần tải model và cài thư viện OCR riêng. Thư viện OCR chạy trong tiến trình riêng, không hạ phiên bản Transformers/Tokenizers của kernel notebook.
Trong lúc cài, log hiển thị gói đang tải/cài; chỉ khi xuất hiện bước nạp DeepSeek-OCR/OCR ảnh thì model mới bắt đầu làm việc.

Giữ model `deepseek-ai/DeepSeek-OCR`, BF16, eager attention và cấu hình OCR từ code đã gửi. Bản này đọc trực tiếp quan hệ ảnh trong DOCX (kể cả preview MathType/VML), giữ vị trí bằng marker, gửi mọi ảnh đọc được và không trắng vào OCR. Không bỏ ảnh vì nhỏ hơn 80 px, bị đoán là hình vẽ hoặc có alt-text. Ảnh trùng chỉ OCR một lần.

WMF/EMF/SVG được chuyển sang PNG bằng LibreOffice Draw. Nếu cần, lần đầu tự cài LibreOffice nên sẽ lâu hơn; nếu chuyển đổi lỗi thì giữ file gốc và báo cần kiểm tra, không âm thầm bỏ ảnh. ZIP có `image-report.json` ghi trạng thái từng ảnh, kết quả OCR thô và số lỗi. Word Equation gốc vẫn do Pandoc đọc trực tiếp, không giả là đã OCR.

Nếu không có ảnh cần OCR, giao diện ghi rõ **không có inference model**, không coi đó là benchmark OCR. Kết quả ZIP có Markdown, ảnh, báo cáo từng ảnh, log và metrics. Mỗi lượt có thư mục riêng, không xóa kết quả cũ.

Đã kiểm tra luồng nhập file/UI và cú pháp local; chưa xác nhận inference GPU của bản notebook này trên Kaggle.

Tham khảo kỹ thuật: [model gốc](https://huggingface.co/deepseek-ai/DeepSeek-OCR), [pip --target](https://pip.pypa.io/en/stable/cli/pip_install/#cmdoption-t).


In [ ]:
# Chạy cell này một lần. Mọi thao tác tiếp theo ở các nút bên dưới.
import importlib.util
from pathlib import Path
import subprocess
import sys

if importlib.util.find_spec("ipywidgets") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "ipywidgets>=8,<9"], check=True)

APP_SOURCE = '"""One-screen UI for the attachment\'s Pandoc + DeepSeek-OCR worker."""\nfrom __future__ import annotations\n\nimport base64\nfrom collections import deque\nfrom datetime import datetime\nimport hashlib\nimport html\nimport importlib.util\nimport io\nimport json\nimport mimetypes\nimport os\nfrom pathlib import Path\nimport queue\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport threading\nimport time\nimport traceback\nfrom uuid import uuid4\nimport zipfile\n\n\nAPP_VERSION = "hybrid-clean-5-live-status"\nPROGRESS_PREFIX = "@@HYBRID_PROGRESS@@"\nHEARTBEAT_SECONDS = 5.0\n# Explicit closure with --no-deps: never download/replace Kaggle\'s torch stack.\nREQUIREMENTS = [\n    "transformers==4.46.3", "tokenizers==0.20.3", "huggingface-hub==0.36.2",\n    "Pillow>=10.1,<13", "numpy>=1.26,<3", "pymupdf>=1.24,<2",\n    "einops>=0.8", "addict>=2.4", "easydict>=1.13", "accelerate>=0.26,<2",\n    "safetensors>=0.4.3", "filelock>=3.16", "packaging>=23",\n    "pyyaml>=6", "regex>=2024", "requests>=2.32,<3", "tqdm>=4.66",\n    "fsspec>=2023.5", "typing-extensions>=4.9", "hf-xet>=1.1.3,<2",\n    "psutil>=5.9", "urllib3>=1.26,<3", "certifi>=2024",\n    "charset-normalizer>=3,<4", "idna>=3,<4", "pypandoc-binary>=1.14,<2",\n]\n\n\nclass InputProblem(ValueError):\n    """An actionable input/environment message, not an inference traceback."""\n\n\nclass SubprocessFailure(RuntimeError):\n    def __init__(self, step, returncode, tail):\n        self.step = step\n        self.returncode = returncode\n        self.tail = tail\n        super().__init__(f"{step} thất bại (mã {returncode}).\\n{tail}")\n\n\nclass JobFailure(RuntimeError):\n    def __init__(self, message, log_path):\n        self.log_path = Path(log_path)\n        super().__init__(message)\n\n\ndef validate_docx(path):\n    if path is None or not str(path).strip():\n        raise InputProblem("Chưa chọn file DOCX.")\n    path = Path(path).expanduser().resolve()\n    if not path.is_file() or path.suffix.lower() != ".docx":\n        raise InputProblem("Không phải file DOCX: " + str(path))\n    try:\n        with zipfile.ZipFile(path) as archive:\n            if not {"[Content_Types].xml", "word/document.xml"}.issubset(archive.namelist()):\n                raise InputProblem("File không có cấu trúc Word DOCX hợp lệ.")\n    except zipfile.BadZipFile as error:\n        raise InputProblem("File bị hỏng hoặc không phải DOCX.") from error\n    return path\n\n\ndef save_upload(value, upload_root):\n    """Handle widget 7/8 values without relying on notebook globals."""\n    entries = list(value.values()) if isinstance(value, dict) else list(value or ())\n    if not entries:\n        return None\n    if len(entries) != 1:\n        raise InputProblem("Bản thử này nhận một DOCX mỗi lần.")\n    item = entries[0]\n    name = item.get("name") or item.get("metadata", {}).get("name", "")\n    name = Path(name.replace("\\\\", "/")).name\n    if not name or Path(name).suffix.lower() != ".docx":\n        raise InputProblem("Hãy chọn file đề .docx, không phải notebook .ipynb.")\n    content = bytes(item["content"])\n    try:\n        with zipfile.ZipFile(io.BytesIO(content)) as archive:\n            if not {"[Content_Types].xml", "word/document.xml"}.issubset(archive.namelist()):\n                raise InputProblem("File tải lên không có cấu trúc Word DOCX hợp lệ.")\n    except zipfile.BadZipFile as error:\n        raise InputProblem("File tải lên bị hỏng hoặc không phải DOCX.") from error\n    upload_root = Path(upload_root)\n    upload_root.mkdir(parents=True, exist_ok=True)\n    destination = upload_root / (hashlib.sha256(content).hexdigest()[:16] + "-" + name)\n    if not destination.is_file():\n        destination.write_bytes(content)\n    return validate_docx(destination)\n\n\ndef discover_docx(roots):\n    found, visited = set(), set()\n    for root in roots:\n        for folder, directories, filenames in os.walk(root, followlinks=True):\n            real_folder = Path(folder).resolve()\n            if real_folder in visited:\n                directories[:] = []\n                continue\n            visited.add(real_folder)\n            for name in filenames:\n                path = Path(folder) / name\n                if path.suffix.lower() == ".docx" and path.is_file():\n                    found.add(path.resolve())\n    return sorted(found)\n\n\ndef gpu_status():\n    try:\n        import torch\n        if not torch.cuda.is_available():\n            return False, "GPU chưa sẵn sàng. Chọn GPU trong Settings → Accelerator rồi khởi động lại phiên nếu cần."\n        free_bytes, _ = torch.cuda.mem_get_info(0)\n        name = torch.cuda.get_device_name(0)\n        if free_bytes < 7_000_000_000:\n            return False, f"{name}: chỉ còn {free_bytes / 1e9:.1f} GB VRAM; code gửi yêu cầu ít nhất 7 GB."\n        return True, f"{name} · {free_bytes / 1e9:.1f} GB VRAM trống"\n    except Exception as error:\n        return False, "Không kiểm tra được GPU: " + str(error)\n\n\ndef child_environment(dependencies):\n    env = os.environ.copy()\n    # A child interpreter sees private dependencies first; the notebook does not.\n    env["PYTHONPATH"] = str(dependencies)\n    env["PYTHONUNBUFFERED"] = "1"\n    env["PYTHONUTF8"] = "1"\n    env["TOKENIZERS_PARALLELISM"] = "false"\n    return env\n\n\ndef stream_command(command, *, env, cwd, log, emit, step="Tiến trình"):\n    tail = deque(maxlen=60)\n    log.write("\\n=== " + step + " ===\\n")\n    log.flush()\n    kwargs = {"creationflags": subprocess.CREATE_NO_WINDOW} if os.name == "nt" else {}\n    process = subprocess.Popen(\n        command, cwd=cwd, env=env, stdout=subprocess.PIPE,\n        stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",\n        bufsize=1, **kwargs,\n    )\n    started = last_output = last_heartbeat = time.monotonic()\n    pending = queue.Queue()\n\n    def read_output():\n        # Only this reader blocks on stdout. The caller can report live state\n        # even while apt/model loading produces no newline for several minutes.\n        try:\n            for line in process.stdout:\n                pending.put(("line", line))\n        except Exception as error:\n            pending.put(("error", error))\n        finally:\n            pending.put(("eof", None))\n\n    def publish(event, **values):\n        line = PROGRESS_PREFIX + json.dumps(dict(event=event, pid=process.pid, **values), ensure_ascii=False)\n        log.write(line + "\\n")\n        log.flush()\n        emit(line)\n\n    reader = threading.Thread(target=read_output, name="hybrid-stdout", daemon=True)\n    reader.start()\n    try:\n        publish("phase", phase=step, elapsed_s=0, silence_s=0)\n        interval = max(0.01, HEARTBEAT_SECONDS)\n        while True:\n            try:\n                kind, value = pending.get(timeout=min(interval, 0.5))\n            except queue.Empty:\n                kind, value = "wait", None\n            if kind == "eof":\n                break\n            if kind == "error":\n                raise value\n            if kind == "line":\n                last_output = time.monotonic()\n                tail.append(value)\n                log.write(value)\n                log.flush()\n                emit(value.rstrip("\\n"))\n            now = time.monotonic()\n            if now - last_heartbeat >= interval and process.poll() is None:\n                publish("heartbeat", elapsed_s=round(now - started, 1), silence_s=round(now - last_output, 1))\n                last_heartbeat = now\n        code = process.wait()\n        publish("exit", returncode=code, elapsed_s=round(time.monotonic() - started, 1))\n    except BaseException:\n        process.terminate()\n        try:\n            process.wait(timeout=10)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n        raise\n    finally:\n        reader.join(timeout=1)\n        process.stdout.close()\n    if code:\n        raise SubprocessFailure(step, code, "".join(tail)[-12000:].strip())\n\n\ndef ensure_dependencies(root, log, emit):\n    key = hashlib.sha256(json.dumps([list(sys.version_info[:2]), REQUIREMENTS]).encode()).hexdigest()[:12]\n    parent = Path(root) / "dependencies"\n    target = parent / key\n    marker = target / "ready.json"\n    if marker.is_file():\n        emit("Dùng lại thư viện OCR đã cài riêng.")\n        return target\n    parent.mkdir(parents=True, exist_ok=True)\n    stage = Path(tempfile.mkdtemp(prefix=key + "-", dir=parent))\n    emit("Lần đầu: cài thư viện OCR riêng, không thay thư viện của notebook.")\n    command = [\n        sys.executable, "-m", "pip", "install", "--progress-bar", "off",\n        "--disable-pip-version-check", "--no-warn-script-location",\n        "--no-warn-conflicts", "--no-deps", "--target", str(stage), *REQUIREMENTS,\n    ]\n    stream_command(command, env=os.environ.copy(), cwd=root, log=log, emit=emit, step="Cài thư viện riêng")\n    # Check real imports/declared runtime requirements inside the child, rather\n    # than trusting the base environment\'s unrelated gradio/litellm metadata.\n    check = r\'\'\'\nimport importlib.metadata as md\nfrom packaging.requirements import Requirement\nfrom packaging.version import Version\nimport transformers, tokenizers, torch, torchvision, PIL, numpy, pypandoc\nfrom transformers import AutoModel, AutoTokenizer, StoppingCriteria, StoppingCriteriaList\nassert transformers.__version__ == "4.46.3"\nassert tokenizers.__version__ == "0.20.3"\nfor package in ("transformers", "tokenizers", "huggingface-hub", "accelerate", "requests", "Pillow", "pypandoc_binary"):\n    for raw in md.requires(package) or []:\n        requirement = Requirement(raw)\n        if requirement.marker and not requirement.marker.evaluate({"extra": ""}):\n            continue\n        version = md.version(requirement.name)\n        if requirement.specifier and Version(version) not in requirement.specifier:\n            raise RuntimeError(f"{package}: {raw}, installed={version}")\nprint("Dependency check OK | torch=" + torch.__version__ + " | transformers=" + transformers.__version__)\n\'\'\'\n    stream_command([sys.executable, "-c", check], env=child_environment(stage), cwd=root, log=log, emit=emit,\n                   step="Kiểm tra import và phiên bản thư viện")\n    (stage / "ready.json").write_text(json.dumps({"requirements": REQUIREMENTS}), encoding="utf-8")\n    if target.exists():\n        # A previous interrupted install is preserved; do not merge its files.\n        return stage\n    stage.rename(target)\n    return target\n\n\ndef run_job(source, root, worker_source, emit=print):\n    """Validate first; only then install dependencies and start real inference."""\n    source = validate_docx(source)\n    ready, message = gpu_status()\n    if not ready:\n        raise InputProblem(message)\n    root = Path(root).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    job = root / "runs" / (datetime.now().strftime("%Y%m%d-%H%M%S-") + uuid4().hex[:8])\n    job.mkdir(parents=True)\n    started = time.perf_counter()\n    copied = job / "input.docx"\n    shutil.copyfile(source, copied)\n    worker = job / "worker.py"\n    worker.write_text(worker_source, encoding="utf-8")\n    log_path = job / "run.log"\n    try:\n        with log_path.open("w", encoding="utf-8") as log:\n            log.write("Source: " + str(source) + "\\n")\n            dependencies = ensure_dependencies(root, log, emit)\n            setup_s = time.perf_counter() - started\n            emit("Đang chạy đúng pipeline Pandoc + DeepSeek-OCR đã gửi…")\n            stream_command(\n                [sys.executable, "-u", str(worker), "--input", str(copied), "--output", str(job / "result"),\n                 "--title", re.sub(r"^[0-9a-f]{16}-", "", source.stem)],\n                env=child_environment(dependencies), cwd=job, log=log, emit=emit,\n                step="Pandoc + DeepSeek-OCR",\n            )\n        results = job / "result"\n        if not (results / "final.md").is_file():\n            raise RuntimeError("Tiến trình kết thúc nhưng chưa tạo final.md: " + str(job))\n        metrics = json.loads((results / "metrics.json").read_text(encoding="utf-8"))\n        metrics["source_name"] = source.name\n        metrics["setup_s"] = setup_s\n        shutil.copyfile(log_path, results / "run.log")\n        metrics["notebook_job_s_before_zip"] = time.perf_counter() - started\n        (results / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")\n        emit(PROGRESS_PREFIX + json.dumps(dict(event="phase", phase="Đóng gói kết quả ZIP", pid=None)))\n        archive = Path(shutil.make_archive(str(job / "ket-qua"), "zip", results))\n        return {"job": job, "results": results, "archive": archive, "metrics": metrics}\n    except Exception as error:\n        try:\n            with log_path.open("a", encoding="utf-8") as log:\n                traceback.print_exc(file=log)\n        except OSError:\n            pass\n        raise JobFailure(str(error), log_path) from error\n\n\nclass HybridApp:\n    def __init__(self, root, worker_source):\n        import ipywidgets as widgets\n        self.root = Path(root)\n        self.root.mkdir(parents=True, exist_ok=True)\n        self.worker_source = worker_source\n        self.selected = None\n        self.source_kind = None\n        self.busy = False\n        self.last_result = None\n        self.outcome = None\n        self.phase = "Chưa chạy"\n        self._started_at = self._phase_started = self._last_output_at = time.monotonic()\n        self._last_ui_update = 0.0\n        self._active_pid = None\n        self._last_log_line = ""\n        self.upload = widgets.FileUpload(accept=".docx", multiple=False, description="Chọn DOCX")\n        self.refresh_button = widgets.Button(description="Làm mới")\n        self.dataset = widgets.Dropdown(description="File có sẵn:", layout=widgets.Layout(width="98%"))\n        self.manual_path = widgets.Text(description="Đường dẫn:", layout=widgets.Layout(width="98%"))\n        self.path_button = widgets.Button(description="Dùng đường dẫn")\n        self.run_button = widgets.Button(description="Chạy OCR", button_style="success", icon="play", disabled=True,\n                                         layout=widgets.Layout(width="300px"))\n        self.status = widgets.HTML()\n        self.gpu = widgets.HTML()\n        self.logs = widgets.Output(layout=widgets.Layout(max_height="360px", overflow="auto"))\n        self.results = widgets.Output()\n        self.upload.observe(self.on_upload, names="value")\n        self.dataset.observe(self.on_dataset, names="value")\n        self.path_button.on_click(self.on_manual_path)\n        self.refresh_button.on_click(self.refresh)\n        self.run_button.on_click(self.on_run)\n        alternative = widgets.Accordion(children=[widgets.VBox([\n            self.dataset, widgets.HBox([self.manual_path, self.path_button]), self.refresh_button,\n        ])], selected_index=None)\n        alternative.set_title(0, "Hoặc chọn DOCX đã có trên Kaggle")\n        self.view = widgets.VBox([\n            widgets.HTML("<h3>DOCX → Pandoc + DeepSeek-OCR</h3><p>Chọn file đề, chờ hiện tên file, rồi bấm <b>Chạy OCR</b>.</p>"\n                         "<small>Bản 5 · trạng thái trực tiếp mỗi 5 giây</small>"),\n            self.gpu, self.upload, alternative, self.status, self.run_button,\n            widgets.HTML("<b>Nhật ký chi tiết</b> (cuộn trong khung bên dưới)"), self.logs, self.results,\n        ])\n        self.restore_selection()\n        self.refresh()\n\n    def restore_selection(self):\n        try:\n            saved = json.loads((self.root / "selected.json").read_text(encoding="utf-8"))\n            self.selected = validate_docx(saved["path"])\n            self.source_kind = "saved"\n        except (OSError, ValueError, KeyError, TypeError):\n            self.selected = None\n\n    def select(self, path, source_kind=None):\n        selected = validate_docx(path)\n        if selected != self.selected:\n            self.outcome = None\n        self.selected = selected\n        if source_kind is not None:\n            self.source_kind = source_kind\n        (self.root / "selected.json").write_text(json.dumps({"path": str(selected)}), encoding="utf-8")\n        self.status.value = "<b>Đã chọn:</b> " + html.escape(selected.name) + f" · {selected.stat().st_size:,} bytes"\n        self.update_ready()\n\n    def update_ready(self):\n        ready, message = gpu_status()\n        self.gpu.value = html.escape(message)\n        self.run_button.disabled = self.busy or self.selected is None or not self.selected.is_file() or not ready\n        if self.busy:\n            self.run_button.description = "Đang chạy — xem trạng thái"\n            self.run_button.icon = "hourglass-half"\n            self.run_button.button_style = "warning"\n        elif self.outcome in {"error", "stopped"}:\n            self.run_button.description = "Thử lại OCR"\n            self.run_button.icon = "repeat"\n            self.run_button.button_style = "danger"\n        else:\n            self.run_button.description = "Chạy OCR lại" if self.outcome else "Chạy OCR"\n            self.run_button.icon = "repeat" if self.outcome else "play"\n            self.run_button.button_style = "success"\n\n    def show_running(self, *, silence_s=None, force=False):\n        if not self.busy:\n            return\n        now = time.monotonic()\n        if not force and now - self._last_ui_update < 0.5:\n            return\n        self._last_ui_update = now\n        silence = max(0, now - self._last_output_at) if silence_s is None else silence_s\n        pid = f" · PID {self._active_pid}" if self._active_pid is not None else ""\n        quiet = ("<br><b>Chưa có log mới. Tiến trình chưa kết thúc; chưa xác nhận tiến độ bên trong.</b>"\n                 if silence >= 30 else "")\n        self.status.value = (\n            \'<div style="border:2px solid #9a6700;padding:10px;background:#fff8dc;color:#111">\'\n            \'<b>ĐANG CHẠY — \' + html.escape(self.phase) + \'</b><br>\'\n            + f"Tổng {now - self._started_at:.0f}s · bước hiện tại {now - self._phase_started:.0f}s{pid}<br>"\n            + f"Log mới nhất cách đây {silence:.0f}s. Trạng thái tiến trình cập nhật mỗi 5 giây."\n            + quiet + \'<br><small>\' + html.escape(self._last_log_line[:220]) + \'</small></div>\'\n        )\n\n    def on_log(self, line):\n        """Separate actual output from process-alive signals; do not invent % progress."""\n        now = time.monotonic()\n        if line.startswith(PROGRESS_PREFIX):\n            try:\n                event = json.loads(line[len(PROGRESS_PREFIX):])\n            except (ValueError, TypeError):\n                event = {}\n            if not isinstance(event, dict):\n                event = {}\n            kind = event.get("event")\n            if kind == "heartbeat":\n                self._active_pid = event.get("pid")\n                self.show_running(silence_s=event.get("silence_s", 0), force=True)\n                self.logs.append_stdout(\n                    f"[ĐANG CHẠY] PID {self._active_pid} · tiến trình {event.get(\'elapsed_s\', 0):.0f}s"\n                    f" · không có log mới {event.get(\'silence_s\', 0):.0f}s\\n"\n                )\n                return\n            if kind == "phase":\n                self.phase = event["phase"]\n                self._active_pid = event.get("pid")\n                self._phase_started = self._last_output_at = now\n                self.show_running(force=True)\n                self.logs.append_stdout("[BƯỚC] " + self.phase + "\\n")\n                return\n            if kind == "exit":\n                self._active_pid = None\n                self.logs.append_stdout(f"[TIẾN TRÌNH KẾT THÚC] mã {event.get(\'returncode\')}\\n")\n                return\n        self.logs.append_stdout(line + "\\n")\n        self._last_output_at = now\n        clean = line.strip()\n        if clean:\n            self._last_log_line = clean\n        phase = None\n        if "đang cài LibreOffice Draw" in clean:\n            phase = "Cài LibreOffice Draw — chưa chạy OCR"\n        elif clean.startswith("LibreOffice Draw:"):\n            phase = clean\n        elif clean.startswith("Bản in gốc:"):\n            phase = clean\n        elif "Đang tạo bản in" in clean:\n            phase = "Tạo bản in / PDF"\n        elif clean.startswith("Lần đầu: cài thư viện"):\n            phase = "Cài thư viện OCR riêng — chưa chạy OCR"\n        match = re.search(r"BUOC\\s+([1-4])\\s*:", clean)\n        if match:\n            phase = {"1": "Bóc DOCX / chuẩn bị ảnh", "2": "Tải / nạp model DeepSeek-OCR",\n                     "3": "Nhận dạng ảnh OCR", "4": "Ghép kết quả OCR"}[match[1]]\n        image = re.match(r"\\[(\\d+)/(\\d+)\\]", clean)\n        if image and self.phase.startswith("Nhận dạng"):\n            phase = f"Nhận dạng ảnh OCR {image[1]}/{image[2]}"\n        if phase and phase != self.phase:\n            self.phase = phase\n            self._phase_started = now\n        self.show_running(force=bool(phase))\n\n    def input_error(self, error):\n        self.selected = None\n        self.source_kind = None\n        self.status.value = html.escape(str(error))\n        self.update_ready()\n\n    def refresh(self, _=None):\n        if self.source_kind in (None, "upload") and self.upload.value:\n            try:\n                uploaded = save_upload(self.upload.value, self.root / "uploads")\n                if uploaded:\n                    self.select(uploaded, "upload")\n            except InputProblem as error:\n                self.input_error(error)\n        candidates = discover_docx([Path("/kaggle/input"), self.root / "uploads"])\n        previous = self.dataset.value\n        self.dataset.options = [("— Chọn DOCX —", "")] + [(str(p), str(p)) for p in candidates]\n        if previous in {str(p) for p in candidates}:\n            self.dataset.value = previous\n        if self.selected is not None:\n            try:\n                self.select(self.selected)\n            except InputProblem:\n                self.selected = None\n        if self.selected is None and len(candidates) == 1:\n            try:\n                self.select(candidates[0], "dataset")\n            except InputProblem as error:\n                self.status.value = html.escape(str(error))\n        if self.selected is None:\n            self.status.value = "Chưa chọn DOCX. Bấm <b>Chọn DOCX</b> ở trên. Nếu file nằm trong dataset, mở mục bên dưới và bấm Làm mới."\n        self.update_ready()\n\n    def on_upload(self, change):\n        if self.busy:\n            return\n        try:\n            selected = save_upload(change["new"], self.root / "uploads")\n            if selected:\n                self.select(selected, "upload")\n        except InputProblem as error:\n            self.input_error(error)\n\n    def on_dataset(self, change):\n        if self.busy or not change["new"]:\n            return\n        try:\n            self.select(change["new"], "dataset")\n        except InputProblem as error:\n            self.input_error(error)\n\n    def on_manual_path(self, _=None):\n        try:\n            self.select(self.manual_path.value.strip(), "manual")\n        except InputProblem as error:\n            self.input_error(error)\n\n    def on_run(self, _=None):\n        if self.busy:\n            return\n        self.logs.clear_output(wait=True)\n        self.results.clear_output(wait=True)\n        try:\n            # Read the upload again to handle a delayed observer callback.\n            if self.source_kind in (None, "upload"):\n                uploaded = save_upload(self.upload.value, self.root / "uploads")\n                if uploaded:\n                    self.select(uploaded, "upload")\n            source = validate_docx(self.selected)\n            ready, message = gpu_status()\n            if not ready:\n                raise InputProblem(message)\n        except InputProblem as error:\n            self.status.value = html.escape(str(error))\n            self.update_ready()\n            return\n        self.busy = True\n        self.outcome = None\n        self.phase = "Chuẩn bị thư viện / môi trường"\n        self._started_at = self._phase_started = self._last_output_at = time.monotonic()\n        self._active_pid = None\n        self._last_log_line = source.name\n        for control in (self.upload, self.dataset, self.manual_path, self.path_button, self.refresh_button, self.run_button):\n            control.disabled = True\n        self.update_ready()\n        self.show_running(force=True)\n        self.last_result = None\n        try:\n            # Do NOT wrap execution in `with self.logs`: live IPython Output\n            # suppresses exceptions in __exit__, then execution continues with\n            # no result. Append logs explicitly and let this try catch errors.\n            result = run_job(\n                source, self.root, self.worker_source,\n                emit=self.on_log,\n            )\n            if not isinstance(result, dict) or not isinstance(result.get("metrics"), dict):\n                raise RuntimeError("Tiến trình không trả về kết quả hợp lệ; chưa có kết quả để hiển thị.")\n            self.last_result = result\n            self.render_success(result)\n            self.outcome = "success"\n            unprocessed = result["metrics"].get("images_unprocessed", 0)\n            if unprocessed:\n                self.status.value = f"<b>Đã tạo kết quả, cần kiểm tra {unprocessed} ảnh/công thức chưa nhận được:</b> " + html.escape(source.name)\n            elif result["metrics"].get("print_export_error"):\n                self.status.value = "Đã tạo dữ liệu OCR nhưng bản in gặp lỗi: " + html.escape(result["metrics"]["print_export_error"])\n            elif result["metrics"].get("print_unplaced_images"):\n                self.status.value = "Đã tạo bản in, cần kiểm tra vị trí ảnh/công thức trong mục khôi phục."\n            else:\n                self.status.value = "<b>Hoàn tất:</b> " + html.escape(source.name)\n            self.status.value += f"<br><b>ĐÃ XONG</b> · Tổng {time.monotonic() - self._started_at:.0f}s. Kết quả ở bên dưới."\n        except (Exception, KeyboardInterrupt) as error:\n            self.outcome = "stopped" if isinstance(error, KeyboardInterrupt) else "error"\n            self.status.value = ("<b>ĐÃ DỪNG — lượt chạy bị ngắt.</b>" if self.outcome == "stopped" else\n                                 "<b>LỖI — tiến trình đã dừng.</b> Lỗi gốc nằm trong log bên dưới.")\n            self.logs.append_stderr(traceback.format_exc())\n            self.logs.append_stderr("\\n" + str(error) + "\\n")\n            # Keep an actionable root message in status as well as the log.\n            self.status.value += "<pre style=\'white-space:pre-wrap\'>" + html.escape(str(error)) + "</pre>"\n            log_path = getattr(error, "log_path", None)\n            if log_path is not None and Path(log_path).is_file():\n                self.render_log_download(Path(log_path))\n        finally:\n            self.busy = False\n            for control in (self.upload, self.dataset, self.manual_path, self.path_button, self.refresh_button):\n                control.disabled = False\n            self.update_ready()\n\n    def render_log_download(self, log_path):\n        from IPython.display import HTML\n        try:\n            encoded = base64.b64encode(log_path.read_bytes()).decode("ascii")\n            self.results.append_display_data(HTML(\n                \'<p><a download="run.log" href="data:text/plain;charset=utf-8;base64,\'\n                + encoded + \'">⬇ Tải log lỗi (run.log)</a></p>\'\n            ))\n            self.logs.append_stdout("Log đầy đủ: " + str(log_path) + "\\n")\n        except OSError as error:\n            self.logs.append_stderr("Không đọc được log: " + str(error) + "\\n")\n\n    def render_success(self, result):\n        from IPython.display import FileLink, HTML, Markdown\n        metrics = result["metrics"]\n        if not metrics["gpu_inference_ran"]:\n            self.results.append_stdout("Không chạy model OCR trong lượt này. Xem số ảnh nguồn và image-report.json để phân biệt không có ảnh với ảnh chưa chuyển được; đây KHÔNG phải benchmark model.\\n")\n        if "images_in_docx" in metrics:\n            self.results.append_stdout(\n                f"Ảnh nguồn: {metrics[\'images_in_docx\']} · ảnh vector: {metrics.get(\'vector_images\', 0)} · "\n                f"Word Equation gốc: {metrics.get(\'native_word_equations\', 0)} · "\n                f"ảnh/công thức cần kiểm tra: {metrics.get(\'images_unprocessed\', 0)}\\n"\n            )\n        self.results.append_stdout(f"Ảnh OCR: {metrics[\'unique_ocr_images\']} · thành công: {metrics[\'ocr_successes\']}\\n")\n        self.results.append_stdout(f"Pandoc/tiền xử lý: {metrics[\'parse_preprocess_s\']:.2f}s · nạp model: {metrics[\'model_load_s\']:.2f}s · OCR: {metrics[\'ocr_s\']:.2f}s\\n")\n        if "print_export_s" in metrics:\n            self.results.append_stdout(f"Xuất bản in/PDF: {metrics[\'print_export_s\']:.2f}s (lần đầu có thể gồm cài LibreOffice).\\n")\n        archive = result["archive"]\n        encoded = base64.b64encode(archive.read_bytes()).decode("ascii")\n        self.results.append_display_data(HTML(\'<p><a download="ket-qua-ocr.zip" href="data:application/zip;base64,\' + encoded + \'">⬇ Tải kết quả ZIP</a></p>\'))\n        self.results.append_display_data(FileLink(str(result["results"] / "final.md")))\n        self.results.append_display_data(FileLink(str(result["results"] / "metrics.json")))\n        report = result["results"] / "image-report.json"\n        if report.is_file():\n            self.results.append_display_data(FileLink(str(report)))\n        exam_html = result["results"] / "de-thi.html"\n        if exam_html.is_file():\n            self.results.append_stdout("Bản in dưới đây dùng ảnh và nội dung gốc theo DOCX. OCR được tách riêng để đối chiếu.\\n")\n            for name, mime, label in (\n                ("de-thi.html", "text/html", "Tải đề in HTML — mở để In / lưu PDF"),\n                ("doi-chieu-ocr.html", "text/html", "Tải bản đối chiếu OCR"),\n                ("de-thi-goc.pdf", "application/pdf", "Tải PDF giữ bố cục Word gốc (không phải OCR sinh)"),\n            ):\n                path = result["results"] / name\n                if path.is_file():\n                    encoded = base64.b64encode(path.read_bytes()).decode("ascii")\n                    self.results.append_display_data(HTML(\'<p><a download="\' + name + \'" href="data:\' + mime + \';base64,\' + encoded + \'">\' + label + \'</a></p>\'))\n            page = exam_html.read_text(encoding="utf-8")\n            self.results.append_display_data(HTML(\n                \'<iframe title="Bản đề in" sandbox="allow-scripts allow-modals allow-downloads" \'\n                \'style="width:100%;height:1000px;border:1px solid #ccc" srcdoc="\' + html.escape(page, quote=True) + \'"></iframe>\'\n            ))\n            return  # Do not show the upscaled OCR rasters as the print preview.\n        text = (result["results"] / "final.md").read_text(encoding="utf-8")\n        def image_data(match):\n            path = (result["results"] / match[2]).resolve()\n            if not path.is_relative_to(result["results"].resolve()) or not path.is_file():\n                return match[0]\n            mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"\n            return "![" + match[1] + "](data:" + mime + ";base64," + base64.b64encode(path.read_bytes()).decode("ascii") + ")"\n        self.results.append_display_data(Markdown(re.sub(r"!\\[([^\\]]*)\\]\\(([^)]+)\\)", image_data, text)))\n\n\ndef launch(root, worker_source):\n    from IPython.display import display\n    app = HybridApp(root, worker_source)\n    display(app.view)\n    return app\n'
WORKER_SOURCE = '"""Worker for the original Pandoc + DeepSeek-OCR attachment.\n\nExecuted in a child process with private dependency paths. Pandoc and the\nDeepSeek model remain; image recovery no longer discards small/vector formulas.\n"""\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport time\nimport types\nimport zipfile\n"""Recover DOCX image positions, rasterize vectors and plan image OCR.\n\nNo OCR model or legacy DOCX reader is imported here. Native OMML remains for\nPandoc; image previews (including MathType/VML) go to the same DeepSeek worker.\n"""\nimport hashlib\nfrom copy import deepcopy\nimport json\nimport os\nfrom pathlib import Path\nimport posixpath\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom urllib.parse import unquote, urlsplit\nfrom uuid import uuid4\nimport xml.etree.ElementTree as ET\nimport zipfile\n\nfrom PIL import Image, ImageOps\n\n\n_W = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"\n_R = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"\n_A = "http://schemas.openxmlformats.org/drawingml/2006/main"\n_V = "urn:schemas-microsoft-com:vml"\n_M = "http://schemas.openxmlformats.org/officeDocument/2006/math"\n_WP = "http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing"\n_IMAGE_MD = re.compile(r"!\\[[^\\]]*\\]\\(([^)]+)\\)(?:\\{[^}]*\\})?")\n_VECTORS = {".wmf", ".emf", ".svg"}\n\n\ndef _length_mm(value):\n    match = re.fullmatch(r"\\s*([\\d.]+)\\s*(pt|mm|cm|in|px)\\s*", value or "", re.IGNORECASE)\n    if not match:\n        return None\n    return float(match[1]) * {"pt": 25.4 / 72, "mm": 1, "cm": 10, "in": 25.4, "px": 25.4 / 96}[match[2].lower()]\n\n\ndef image_geometry(node, parents):\n    """Record display geometry separately from raster size used by OCR."""\n    ancestors = []\n    current = node\n    while current is not None:\n        ancestors.append(current)\n        current = parents.get(current)\n    result = dict(width_mm=None, height_mm=None, inline=True, crop={}, baseline_shift_pt=0)\n    drawing = next((item for item in ancestors if item.tag == "{" + _W + "}drawing"), None)\n    if drawing is not None:\n        extent = drawing.find(".//{" + _WP + "}extent")\n        if extent is None:\n            extent = drawing.find(".//{" + _A + "}ext")\n        if extent is not None:\n            try:\n                result.update(width_mm=float(extent.get("cx")) / 36000,\n                              height_mm=float(extent.get("cy")) / 36000)\n            except (ValueError, TypeError):\n                pass\n        result["inline"] = drawing.find(".//{" + _WP + "}anchor") is None\n        rectangle = drawing.find(".//{" + _A + "}srcRect")\n        if rectangle is not None:\n            for side in ("l", "r", "t", "b"):\n                try:\n                    result["crop"][side] = max(0, min(0.99, float(rectangle.get(side, "0")) / 100000))\n                except ValueError:\n                    pass\n    shape = next((item for item in ancestors if item.tag == "{" + _V + "}shape"), None)\n    if shape is not None:\n        styles = dict(part.split(":", 1) for part in shape.get("style", "").split(";") if ":" in part)\n        styles = {key.strip().lower(): value.strip() for key, value in styles.items()}\n        result["width_mm"] = _length_mm(styles.get("width")) or result["width_mm"]\n        result["height_mm"] = _length_mm(styles.get("height")) or result["height_mm"]\n        if styles.get("position") == "absolute":\n            result["inline"] = False\n    run = next((item for item in ancestors if item.tag == "{" + _W + "}r"), None)\n    if run is not None:\n        position = run.find("{" + _W + "}rPr/{" + _W + "}position")\n        if position is not None:\n            try:\n                result["baseline_shift_pt"] = float(position.get("{" + _W + "}val")) / 2\n            except (ValueError, TypeError):\n                pass\n    return result\n\n\ndef mark_docx_images(source, work_dir):\n    """Insert text markers next to image runs so Pandoc cannot lose placement."""\n    source, work_dir = Path(source), Path(work_dir)\n    raw_dir = work_dir / "source_images"\n    raw_dir.mkdir(parents=True, exist_ok=True)\n    records, by_member, changed = [], {}, {}\n    prefix = "HYBRIDIMAGE" + uuid4().hex[:10].upper()\n    marker_index, native_equations = 0, 0\n    for short, uri in (("w", _W), ("r", _R), ("a", _A), ("v", _V), ("m", _M)):\n        ET.register_namespace(short, uri)\n\n    with zipfile.ZipFile(source) as archive:\n        members = set(archive.namelist())\n        parts = sorted(n for n in members if n.startswith("word/") and n.endswith((".xml", ".vml")))\n        for part in parts:\n            try:\n                root = ET.fromstring(archive.read(part))\n            except ET.ParseError:\n                continue\n            native_equations += sum(1 for node in root.iter("{" + _M + "}oMath"))\n            rels_name = posixpath.join(posixpath.dirname(part), "_rels", posixpath.basename(part) + ".rels")\n            relationships = {}\n            if rels_name in members:\n                try:\n                    rels = ET.fromstring(archive.read(rels_name))\n                    relationships = {node.get("Id"): node.attrib for node in rels}\n                except ET.ParseError:\n                    pass\n            parents = {child: parent for parent in root.iter() for child in parent}\n            modified = False\n            for node in list(root.iter()):\n                rid = None\n                kind = "image"\n                if node.tag == "{" + _A + "}blip":\n                    rid = node.get("{" + _R + "}embed") or node.get("{" + _R + "}link")\n                elif node.tag == "{" + _V + "}imagedata":\n                    rid = node.get("{" + _R + "}id")\n                elif node.tag.rsplit("}", 1)[-1] == "OLEObject":\n                    # A MathType object\'s preview is processed as an image.\n                    # A binary with no preview must instead be reported explicitly.\n                    container = parents.get(node)\n                    while container is not None and container.tag != "{" + _W + "}object":\n                        container = parents.get(container)\n                    if container is not None and any(\n                        item.tag in {"{" + _V + "}imagedata", "{" + _A + "}blip"}\n                        for item in container.iter()\n                    ):\n                        continue\n                    rid = node.get("{" + _R + "}id")\n                    kind = "ole_no_preview"\n                if not rid:\n                    continue\n                relationship = relationships.get(rid, {})\n                target = unquote(relationship.get("Target", ""))\n                external = relationship.get("TargetMode") == "External"\n                member = (posixpath.normpath(target.lstrip("/")) if target.startswith("/")\n                          else posixpath.normpath(posixpath.join(posixpath.dirname(part), target)))\n                key = member if target and not external else part + ":" + rid\n                if key not in by_member:\n                    asset_id = "image-" + hashlib.sha256(key.encode()).hexdigest()[:16]\n                    extension = Path(member).suffix.lower()\n                    if not re.fullmatch(r"\\.[a-z0-9]{1,8}", extension):\n                        extension = ".bin"\n                    record = dict(id=asset_id, member=member, kind=kind, extension=extension,\n                                  markers=[], occurrences=[], references=0, parts=[], status="extracted", error="")\n                    if external:\n                        record.update(status="external_unsupported", error="Ảnh liên kết ngoài, không có dữ liệu trong DOCX.")\n                    elif not target or member.startswith("../") or member not in members:\n                        record.update(status="missing_media", error="Không tìm thấy dữ liệu ảnh trong gói DOCX.")\n                    else:\n                        data = archive.read(member)\n                        path = raw_dir / (asset_id + extension)\n                        path.write_bytes(data)\n                        record.update(source_path=str(path), source_sha256=hashlib.sha256(data).hexdigest())\n                    records.append(record)\n                    by_member[key] = record\n                record = by_member[key]\n                geometry = image_geometry(node, parents)\n                record["references"] += 1\n                if part not in record["parts"]:\n                    record["parts"].append(part)\n                # Pandoc may ignore w:t inside a drawing run. Use a separate\n                # text run immediately before the image, splitting preceding\n                # run content so inline image positions are not shifted.\n                anchor = node\n                parent = parents.get(anchor)\n                while parent is not None and parent.tag != "{" + _W + "}r":\n                    anchor, parent = parent, parents.get(parent)\n                if parent is not None and parents.get(parent) is not None:\n                    container = parents[parent]\n                    preceding = [item for item in list(parent)[:list(parent).index(anchor)]\n                                 if item.tag != "{" + _W + "}rPr"]\n                    if preceding:\n                        before_run = ET.Element("{" + _W + "}r")\n                        properties = parent.find("{" + _W + "}rPr")\n                        if properties is not None:\n                            before_run.append(deepcopy(properties))\n                        for item in preceding:\n                            parent.remove(item)\n                            before_run.append(item)\n                            parents[item] = before_run\n                        container.insert(list(container).index(parent), before_run)\n                        parents[before_run] = container\n                    following = [item for item in list(parent)[list(parent).index(anchor) + 1:]\n                                 if item.tag != "{" + _W + "}rPr"]\n                    if following:\n                        after_run = ET.Element("{" + _W + "}r")\n                        properties = parent.find("{" + _W + "}rPr")\n                        if properties is not None:\n                            after_run.append(deepcopy(properties))\n                        for item in following:\n                            parent.remove(item)\n                            after_run.append(item)\n                            parents[item] = after_run\n                        container.insert(list(container).index(parent) + 1, after_run)\n                        parents[after_run] = container\n                    marker_index += 1\n                    marker = prefix + str(marker_index) + "END"\n                    marker_run = ET.Element("{" + _W + "}r")\n                    text = ET.SubElement(marker_run, "{" + _W + "}t")\n                    text.text = marker\n                    container.insert(list(container).index(parent), marker_run)\n                    record["markers"].append(marker)\n                    record["occurrences"].append(dict(marker=marker, **geometry))\n                    modified = True\n            if modified:\n                changed[part] = ET.tostring(root, encoding="utf-8", xml_declaration=True)\n        marked = work_dir / "pandoc-input.docx"\n        with zipfile.ZipFile(marked, "w", compression=zipfile.ZIP_DEFLATED) as output:\n            for info in archive.infolist():\n                output.writestr(info, changed.get(info.filename, archive.read(info.filename)))\n    return dict(docx=marked, images=records, native_word_equations=native_equations)\n\n\ndef _markdown_path(ref, base):\n    ref = ref.strip().strip("<>")\n    if urlsplit(ref).scheme and not re.match(r"^[A-Za-z]:[/\\\\]", ref):\n        return None\n    path = Path(unquote(ref.replace("\\\\ ", " ")))\n    return (path if path.is_absolute() else Path(base) / path).resolve()\n\n\ndef render_vectors(records, output_dir, allow_install=True):\n    """Batch LibreOffice conversion; every missing/failed render is returned."""\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    soffice = shutil.which("soffice")\n    if not soffice and allow_install and sys.platform.startswith("linux") and shutil.which("apt-get"):\n        print("Ảnh WMF/EMF/SVG: đang cài LibreOffice Draw lần đầu…", flush=True)\n        try:\n            install_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive", APT_LISTCHANGES_FRONTEND="none")\n            apt = ["apt-get", "-o", "Dpkg::Use-Pty=0", "-o", "DPkg::Lock::Timeout=60",\n                   "-o", "Acquire::Retries=1", "-o", "Acquire::http::Timeout=30", "-o", "Acquire::https::Timeout=30"]\n            print("LibreOffice Draw: cập nhật danh sách gói (giới hạn 240s)…", flush=True)\n            subprocess.run([*apt, "update"], check=True, timeout=240, env=install_env, stdin=subprocess.DEVNULL)\n            print("LibreOffice Draw: tải / cài gói (giới hạn 480s)…", flush=True)\n            subprocess.run([*apt, "install", "-y", "--no-install-recommends", "libreoffice-core", "libreoffice-draw"],\n                           check=True, timeout=480, env=install_env, stdin=subprocess.DEVNULL)\n            soffice = shutil.which("soffice")\n            if soffice:\n                print("LibreOffice Draw: đã cài xong, bắt đầu chuyển ảnh vector.", flush=True)\n        except (subprocess.SubprocessError, OSError) as error:\n            print("Không cài được LibreOffice:", error, flush=True)\n    if not soffice:\n        return {}, {record["id"]: "Không có LibreOffice để chuyển ảnh vector sang PNG." for record in records}\n    converted, errors = {}, {}\n    unique = {}\n    for record in records:\n        unique.setdefault(record["source_sha256"], []).append(record)\n    entries = list(unique.values())\n    for offset in range(0, len(entries), 32):\n        group = entries[offset:offset + 32]\n        batch = output_dir / ("batch-" + str(offset))\n        batch.mkdir(exist_ok=True)\n        sources = [Path(items[0]["source_path"]) for items in group]\n        print(f"LibreOffice Draw: đang chuyển ảnh vector {offset + 1}–{offset + len(group)}/{len(entries)}…", flush=True)\n        profile = (batch / "profile").resolve().as_uri()\n        export_filter = \'png:draw_png_Export:{"PixelWidth":{"type":"long","value":"1600"},"Translucent":{"type":"boolean","value":"false"}}\'\n        try:\n            process = subprocess.run(\n                [soffice, "-env:UserInstallation=" + profile, "--headless", "--norestore",\n                 "--convert-to", export_filter, "--outdir", str(batch), *map(str, sources)],\n                capture_output=True, text=True, timeout=180,\n            )\n            detail = (process.stdout + "\\n" + process.stderr).strip()\n            (batch / "conversion.log").write_text(detail, encoding="utf-8")\n        except (subprocess.SubprocessError, OSError) as error:\n            detail = str(error)\n        for items, source in zip(group, sources):\n            raster = batch / (source.stem + ".png")\n            for record in items:\n                if raster.is_file():\n                    converted[record["id"]] = raster\n                else:\n                    errors[record["id"]] = "Chuyển vector thất bại: " + detail[-1200:]\n    return converted, errors\n\n\ndef _normalise_image(source, destination, display_destination=None):\n    with Image.open(source) as original:\n        dpi = original.info.get("dpi", (96, 96))\n        rgba = original.convert("RGBA")\n        canvas = Image.new("RGBA", rgba.size, "white")\n        canvas.alpha_composite(rgba)\n        image = canvas.convert("RGB")\n    if display_destination is not None:\n        if not isinstance(dpi, (tuple, list)) or len(dpi) < 2 or not all(isinstance(value, (int, float)) and value > 0 for value in dpi[:2]):\n            dpi = (96, 96)\n        # Uncropped/unscaled pixels are for printing; OCR gets its own raster.\n        image.save(display_destination, format="PNG", dpi=tuple(dpi[:2]))\n    bbox = image.convert("L").point(lambda value: 255 if value < 250 else 0).getbbox()\n    if bbox is None:\n        return None, None\n    image = ImageOps.expand(image.crop(bbox), border=12, fill="white")\n    longest = max(image.size)\n    target = max(384, min(1600, longest))\n    if longest != target:\n        scale = target / longest\n        image = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))), Image.Resampling.LANCZOS)\n    key = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()\n    image.save(destination, format="PNG")\n    return key, list(image.size)\n\n\ndef prepare_image_plan(markdown, extraction, work_dir, vector_converter=None, allow_install=True):\n    """Send every nonblank decodable image to OCR, deduplicated by pixels."""\n    work_dir = Path(work_dir)\n    records = extraction["images"]\n    source_hashes = {}\n    for record in records:\n        if record.get("source_sha256"):\n            source_hashes.setdefault(record["source_sha256"], []).append(record)\n        record["placed"] = False\n    matched_hashes = {}\n    for match in _IMAGE_MD.finditer(markdown):\n        path = _markdown_path(match[1], work_dir)\n        if path is not None and path.is_file():\n            digest = hashlib.sha256(path.read_bytes()).hexdigest()\n            matched_hashes[match[0]] = digest\n            if digest not in source_hashes:\n                record = dict(id="pandoc-" + digest[:16], member=None, kind="image",\n                              extension=path.suffix.lower(), markers=[], references=1, parts=[],\n                              status="extracted", error="", source_path=str(path), source_sha256=digest, placed=False)\n                records.append(record)\n                source_hashes[digest] = [record]\n\n    def replace_pandoc_image(match):\n        candidates = source_hashes.get(matched_hashes.get(match[0]), [])\n        if not candidates:\n            return match[0]\n        if any(marker in markdown for record in candidates for marker in record["markers"]):\n            return ""  # A marker supplies the same image at its source position.\n        record = candidates[0]\n        record["placed"] = True\n        return "![](" + record["source_path"] + ")"\n    result = _IMAGE_MD.sub(replace_pandoc_image, markdown)\n    for record in records:\n        ref = record.get("source_path")\n        placeholder = ("![](" + ref + ")" if ref else "[Ảnh nguồn bị thiếu: " + record["id"] + "]")\n        for marker in record["markers"]:\n            if marker in result:\n                result = result.replace(marker, "\\n\\n" + placeholder + "\\n\\n")\n                record["placed"] = True\n        if not record["placed"]:\n            result += "\\n\\n### Ảnh/công thức khôi phục từ DOCX — chưa gắn được vị trí\\n\\n" + placeholder + "\\n"\n            record["placement_warning"] = "Pandoc không giữ vị trí; ảnh được đưa vào mục khôi phục riêng."\n\n    vectors = [record for record in records if record["extension"] in _VECTORS and record.get("source_path")]\n    converter = vector_converter or render_vectors\n    converted, vector_errors = converter(vectors, work_dir / "converted", allow_install=allow_install) if vectors else ({}, {})\n    prepared = work_dir / "formula_png"\n    prepared.mkdir(exist_ok=True)\n    display_images = work_dir / "display_images"\n    display_images.mkdir(exist_ok=True)\n    tasks, ref_to_hash, seen = [], {}, {}\n    for record in records:\n        if record["kind"] == "ole_no_preview":\n            record.update(status="ole_no_preview", error="Đối tượng OLE không có ảnh xem trước; không thể đưa tệp nhị phân vào OCR.")\n        elif record.get("source_path"):\n            source = Path(record["source_path"])\n            if record["extension"] in _VECTORS:\n                source = converted.get(record["id"])\n                if source is None:\n                    record.update(status="conversion_error", error=vector_errors.get(record["id"], "Thiếu ảnh PNG đã chuyển đổi."))\n            if source is not None:\n                destination = prepared / (record["id"] + ".png")\n                display_destination = display_images / (record["id"] + ".png")\n                try:\n                    key, size = _normalise_image(source, destination, display_destination)\n                    if key is None:\n                        if record["extension"] in _VECTORS:\n                            record.update(status="conversion_error", error="Ảnh PNG chuyển từ vector bị trắng; giữ vector gốc để kiểm tra.")\n                        else:\n                            record["status"] = "blank"\n                    else:\n                        record.update(pixel_key=key, raster_path=str(destination), raster_size=size, status="queued")\n                        record["display_path"] = str(display_destination)\n                        ref_to_hash[str(destination)] = key\n                        if key in seen:\n                            record["duplicate_of"] = seen[key]\n                        else:\n                            seen[key] = record["id"]\n                            tasks.append((str(destination), destination, key))\n                except Exception as error:\n                    record.update(status="decode_error", error=str(error))\n        original_ref = record.get("source_path")\n        if original_ref:\n            original_tag = "![](" + original_ref + ")"\n            if record.get("raster_path"):\n                result = result.replace(original_tag, "![](" + record["raster_path"] + ")")\n            elif record["status"] != "blank":\n                warning = "[Chưa nhận dạng được ảnh/công thức " + record["id"] + ": " + record.get("error", "") + "](" + original_ref + ")"\n                result = result.replace(original_tag, warning)\n    plan = dict(markup=result, records=records, tasks=tasks, ref_to_hash=ref_to_hash,\n                native_word_equations=extraction.get("native_word_equations", 0))\n    write_image_report(plan, work_dir)\n    return plan\n\n\ndef record_ocr_result(plan, key, raw, cleaned):\n    for record in plan["records"]:\n        if record.get("pixel_key") == key:\n            record.update(status="ocr_success" if cleaned else "ocr_empty", raw_ocr=raw, ocr_text=cleaned)\n\n\ndef write_image_report(plan, work_dir):\n    records = plan["records"]\n    error_states = {"conversion_error", "decode_error", "missing_media", "external_unsupported", "ole_no_preview", "ocr_empty"}\n    summary = {\n        "images_in_docx": sum(record["kind"] == "image" for record in records),\n        "image_occurrences": sum(record["references"] for record in records),\n        "native_word_equations": plan["native_word_equations"],\n        "queued_unique_images": len(plan["tasks"]),\n        "duplicate_images": sum(bool(record.get("duplicate_of")) for record in records),\n        "blank_images": sum(record["status"] == "blank" for record in records),\n        "vector_images": sum(record["extension"] in _VECTORS for record in records),\n        "conversion_errors": sum(record["status"] == "conversion_error" for record in records),\n        "images_unprocessed": sum(record["status"] in error_states for record in records),\n    }\n    plan["summary"] = summary\n    report = dict(summary=summary, images=records)\n    serialized = json.dumps(report, ensure_ascii=False, indent=2)\n    serialized = serialized.replace(str(Path(work_dir)) + "/", "")\n    (Path(work_dir) / "image-report.json").write_text(serialized, encoding="utf-8")\n    return summary\n\n"""Self-contained print HTML at DOCX image sizes, plus a native-layout PDF.\n\nThe print document uses source content/images. OCR annotations are a separately\nlabelled comparison layer, not a claim that recognition reproduced the source.\n"""\nimport base64\nfrom collections import Counter\nimport hashlib\nimport html\nfrom html.parser import HTMLParser\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nfrom urllib.parse import unquote, urlsplit\nimport xml.etree.ElementTree as ET\nimport zipfile\n\nfrom PIL import Image\n\n\n_PRINT_W = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"\n\n\ndef read_print_setup(source):\n    setup = dict(width_mm=210.0, height_mm=297.0, top_mm=15.0, right_mm=15.0,\n                 bottom_mm=15.0, left_mm=15.0, font="Times New Roman", font_pt=12.0,\n                 columns=1, section_count=1)\n    with zipfile.ZipFile(source) as archive:\n        root = ET.fromstring(archive.read("word/document.xml"))\n        sections = list(root.iter("{" + _PRINT_W + "}sectPr"))\n        setup["section_count"] = max(1, len(sections))\n        if sections:\n            section = sections[-1]\n            page = section.find("{" + _PRINT_W + "}pgSz")\n            margins = section.find("{" + _PRINT_W + "}pgMar")\n            for element, names in ((page, {"w": "width_mm", "h": "height_mm"}),\n                                   (margins, {"top": "top_mm", "right": "right_mm", "bottom": "bottom_mm", "left": "left_mm"})):\n                if element is not None:\n                    for attribute, key in names.items():\n                        try:\n                            setup[key] = float(element.get("{" + _PRINT_W + "}" + attribute)) * 25.4 / 1440\n                        except (ValueError, TypeError):\n                            pass\n            columns = section.find("{" + _PRINT_W + "}cols")\n            if columns is not None:\n                try:\n                    setup["columns"] = max(1, min(4, int(columns.get("{" + _PRINT_W + "}num", "1"))))\n                except ValueError:\n                    pass\n        if "word/styles.xml" in archive.namelist():\n            styles = ET.fromstring(archive.read("word/styles.xml"))\n            properties = []\n            for style in styles.findall("{" + _PRINT_W + "}style"):\n                if style.get("{" + _PRINT_W + "}styleId", "").lower() == "normal":\n                    props = style.find("{" + _PRINT_W + "}rPr")\n                    if props is not None:\n                        properties.append(props)\n            default = styles.find("{" + _PRINT_W + "}docDefaults/{" + _PRINT_W + "}rPrDefault/{" + _PRINT_W + "}rPr")\n            if default is not None:\n                properties.insert(0, default)\n            for props in properties:\n                fonts = props.find("{" + _PRINT_W + "}rFonts")\n                size = props.find("{" + _PRINT_W + "}sz")\n                if fonts is not None:\n                    setup["font"] = fonts.get("{" + _PRINT_W + "}ascii") or fonts.get("{" + _PRINT_W + "}hAnsi") or setup["font"]\n                if size is not None:\n                    try:\n                        setup["font_pt"] = max(8, min(24, float(size.get("{" + _PRINT_W + "}val")) / 2))\n                    except (ValueError, TypeError):\n                        pass\n        font_weights, size_weights, text_weight = Counter(), Counter(), 0\n        for run in root.iter("{" + _PRINT_W + "}r"):\n            weight = len("".join(node.text or "" for node in run.iter("{" + _PRINT_W + "}t")))\n            if not weight:\n                continue\n            text_weight += weight\n            props = run.find("{" + _PRINT_W + "}rPr")\n            if props is None:\n                continue\n            fonts = props.find("{" + _PRINT_W + "}rFonts")\n            size = props.find("{" + _PRINT_W + "}sz")\n            if fonts is not None:\n                family = fonts.get("{" + _PRINT_W + "}ascii") or fonts.get("{" + _PRINT_W + "}hAnsi")\n                if family:\n                    font_weights[family] += weight\n            if size is not None:\n                try:\n                    size_weights[float(size.get("{" + _PRINT_W + "}val")) / 2] += weight\n                except (TypeError, ValueError):\n                    pass\n        if font_weights and sum(font_weights.values()) >= text_weight / 2:\n            setup["font"] = font_weights.most_common(1)[0][0]\n        if size_weights and sum(size_weights.values()) >= text_weight / 2:\n            setup["font_pt"] = max(8, min(24, size_weights.most_common(1)[0][0]))\n    return setup\n\n\ndef _local_print_path(value, root):\n    value = unquote(value.strip().strip("<>"))\n    if urlsplit(value).scheme and not re.match(r"^[A-Za-z]:[/\\\\]", value):\n        return None\n    path = Path(value)\n    return (path if path.is_absolute() else Path(root) / path).resolve()\n\n\ndef _image_source(record):\n    path = record.get("display_path")\n    if path and Path(path).is_file():\n        return Path(path)\n    path = record.get("source_path")\n    if path and Path(path).is_file() and Path(path).suffix.lower() in {".png", ".jpg", ".jpeg", ".gif", ".bmp", ".webp"}:\n        return Path(path)\n    return None\n\n\ndef source_image_html(record, geometry=None):\n    """Use Word\'s physical box, never the upscaled OCR PNG\'s pixel dimensions."""\n    geometry = geometry or {}\n    source = _image_source(record)\n    if source is None:\n        return \'<span class="missing-image">[Ảnh/công thức cần kiểm tra: \' + html.escape(record["id"]) + \']</span>\'\n    with Image.open(source) as image:\n        width_px, height_px = image.size\n        dpi = image.info.get("dpi", (96, 96))\n    if not isinstance(dpi, (tuple, list)) or len(dpi) < 2 or not all(isinstance(v, (int, float)) and v > 0 for v in dpi[:2]):\n        dpi = (96, 96)\n    width = geometry.get("width_mm") or width_px / dpi[0] * 25.4\n    height = geometry.get("height_mm") or height_px / dpi[1] * 25.4\n    if geometry.get("width_mm") and not geometry.get("height_mm"):\n        height = width * height_px / max(1, width_px)\n    if geometry.get("height_mm") and not geometry.get("width_mm"):\n        width = height * width_px / max(1, height_px)\n    width, height = max(0.1, width), max(0.1, height)\n    crop = geometry.get("crop", {})\n    left, right, top, bottom = [crop.get(key, 0) for key in ("l", "r", "t", "b")]\n    inner_style = ""\n    if any((left, right, top, bottom)):\n        horizontal = max(0.01, 1 - left - right)\n        vertical = max(0.01, 1 - top - bottom)\n        inner_style = (f"left:{-left / horizontal * 100:.5f}%;top:{-top / vertical * 100:.5f}%;"\n                       f"width:{100 / horizontal:.5f}%;height:{100 / vertical:.5f}%;object-fit:fill")\n    data = base64.b64encode(source.read_bytes()).decode("ascii")\n    mime = "image/png" if source.suffix.lower() == ".png" else "image/" + ("jpeg" if source.suffix.lower() in {".jpg", ".jpeg"} else source.suffix[1:].lower())\n    shift = geometry.get("baseline_shift_pt", 0)\n    alignment = f"{shift:g}pt" if shift else "middle"\n    style = f"width:{width:.5f}mm;aspect-ratio:{width:.8f}/{height:.8f};vertical-align:{alignment}"\n    return (\'<span class="source-image" data-image-id="\' + html.escape(record["id"], quote=True)\n            + f\'" data-width-mm="{width:.5f}" data-height-mm="{height:.5f}" style="{style}">\'\n            + \'<img alt="Ảnh/công thức gốc" style="\' + inner_style + \'" src="data:\' + mime + \';base64,\' + data + \'"></span>\')\n\n\nclass _OCRFragments(HTMLParser):\n    def __init__(self):\n        super().__init__(convert_charrefs=False)\n        self.fragments, self.current, self.depth, self.buffer = {}, None, 0, []\n\n    def handle_starttag(self, tag, attrs):\n        attrs = dict(attrs)\n        if self.current is None and tag == "div" and attrs.get("id", "").startswith("ocr-"):\n            self.current, self.depth, self.buffer = attrs["id"][4:], 1, []\n            return\n        if self.current is not None:\n            if tag == "div":\n                self.depth += 1\n            self.buffer.append(self.get_starttag_text())\n\n    def handle_endtag(self, tag):\n        if self.current is not None:\n            if tag == "div":\n                self.depth -= 1\n                if not self.depth:\n                    self.fragments[self.current] = "".join(self.buffer)\n                    self.current = None\n                    return\n            self.buffer.append("</" + tag + ">")\n\n    def handle_startendtag(self, tag, attrs):\n        if self.current is not None:\n            self.buffer.append(self.get_starttag_text())\n\n    def handle_data(self, data):\n        if self.current is not None:\n            self.buffer.append(html.escape(data, quote=False))\n\n    def handle_entityref(self, name):\n        if self.current is not None:\n            self.buffer.append("&" + name + ";")\n\n    def handle_charref(self, name):\n        if self.current is not None:\n            self.buffer.append("&#" + name + ";")\n\n\ndef _render_ocr_fragments(records, pandoc, output_dir):\n    blocks = []\n    for record in records:\n        if record.get("ocr_text"):\n            blocks.append("::: {#ocr-" + record["id"] + "}\\n" + record["ocr_text"] + "\\n:::")\n    if not blocks:\n        return {}\n    process = subprocess.run(\n        [pandoc, "-f", "markdown+fenced_divs+tex_math_dollars-raw_html", "-t", "html5", "--mathml", "--wrap=none"],\n        input="\\n\\n".join(blocks), capture_output=True, text=True, encoding="utf-8", timeout=60,\n    )\n    if process.returncode:\n        return {}\n    parser = _OCRFragments()\n    parser.feed(process.stdout)\n    return parser.fragments\n\n\nclass _ExamBody(HTMLParser):\n    def __init__(self, source_html, records, root, fragments):\n        super().__init__(convert_charrefs=False)\n        self.root, self.fragments, self.output, self.placed = Path(root), fragments, [], set()\n        self.records, self.by_digest, self.markers = records, {}, {}\n        for record in records:\n            if record.get("source_sha256"):\n                self.by_digest.setdefault(record["source_sha256"], []).append(record)\n            for occurrence in record.get("occurrences", []):\n                self.markers[occurrence["marker"]] = (record, occurrence)\n        self.present_markers = {marker for marker in self.markers if marker in source_html}\n        self.pattern = re.compile("(" + "|".join(re.escape(m) for m in sorted(self.markers, key=len, reverse=True)) + ")") if self.markers else None\n\n    def image(self, record, geometry=None):\n        self.placed.add(record["id"])\n        return source_image_html(record, geometry)\n\n    def handle_starttag(self, tag, attrs):\n        if tag == "img":\n            ref = dict(attrs).get("src", "")\n            path = _local_print_path(ref, self.root)\n            if path is not None and path.is_file():\n                digest = hashlib.sha256(path.read_bytes()).hexdigest()\n                candidates = self.by_digest.get(digest, [])\n                if candidates:\n                    if any(marker in self.present_markers for record in candidates for marker in record.get("markers", [])):\n                        return\n                    record = candidates[0]\n                    self.output.append(self.image(record, (record.get("occurrences") or [{}])[0]))\n                    return\n        self.output.append(self.get_starttag_text())\n\n    def handle_startendtag(self, tag, attrs):\n        self.handle_starttag(tag, attrs)\n\n    def handle_endtag(self, tag):\n        if tag not in {"img", "br", "hr", "meta", "link", "input", "wbr", "source"}:\n            self.output.append("</" + tag + ">")\n\n    def handle_data(self, data):\n        if not self.pattern:\n            self.output.append(html.escape(data, quote=False))\n            return\n        for part in self.pattern.split(data):\n            if part in self.markers:\n                record, geometry = self.markers[part]\n                self.output.append(self.image(record, geometry))\n            else:\n                self.output.append(html.escape(part, quote=False))\n\n    def handle_entityref(self, name):\n        self.output.append("&" + name + ";")\n\n    def handle_charref(self, name):\n        self.output.append("&#" + name + ";")\n\n\ndef _page_html(title, body, setup, comparison=False, unplaced="", ocr_notes=""):\n    font = re.sub(r"[^\\w \\-]", "", setup["font"])\n    content_width = max(50, setup["width_mm"] - setup["left_mm"] - setup["right_mm"])\n    css = f\'\'\'\n@page{{size:{setup[\'width_mm\']:.3f}mm {setup[\'height_mm\']:.3f}mm;margin:{setup[\'top_mm\']:.3f}mm {setup[\'right_mm\']:.3f}mm {setup[\'bottom_mm\']:.3f}mm {setup[\'left_mm\']:.3f}mm}}\n*{{box-sizing:border-box}}body{{margin:0;background:#e9ecef;color:#000;font:{setup[\'font_pt\']:g}pt/1.35 "{font}","Times New Roman",serif}}\n.toolbar{{position:sticky;top:0;z-index:5;padding:12px;background:#fff;border-bottom:1px solid #bbb;font:14px/1.5 Arial,sans-serif}}\n.toolbar button{{padding:7px 14px;margin-right:15px;cursor:pointer}}.toolbar small{{display:block;color:#444}}\n.paper{{width:{content_width:.3f}mm;max-width:100%;margin:20px auto;padding:0;background:white;box-shadow:0 1px 10px #bbb}}\n.paper-inner{{padding:0;overflow-wrap:break-word}}p{{margin:0 0 5pt}}h1,h2,h3{{break-after:avoid;margin:9pt 0 5pt}}h1{{font-size:16pt}}h2{{font-size:14pt}}h3{{font-size:1em}}\ntable{{border-collapse:collapse;max-width:100%;margin:5pt 0}}td,th{{padding:3pt 5pt;vertical-align:top}}tr{{break-inside:avoid}}img{{max-width:100%}}\n.source-image{{position:relative;display:inline-block;max-width:100%;overflow:hidden;break-inside:avoid;line-height:0}}\n.source-image>img{{position:absolute;inset:0;width:100%;height:100%;max-width:none;object-fit:fill}}\nmath{{font-size:1em}}.ocr-notes{{display:none;background:#fff9dc;border-top:2px solid #cc9;padding:8pt;margin-top:15pt;line-height:1.3}}\n.show-ocr .ocr-notes{{display:block}}.ocr-notes pre{{white-space:pre-wrap;font-size:.9em}}\n.missing-image{{color:#a00;border:1px dashed #c77;padding:2pt;font-size:.9em}}.recovered{{border-top:1px dashed #aaa;margin-top:15pt}}\n@media screen{{.paper{{padding:12mm;box-sizing:content-box}}}}\n@media print{{body{{background:white}}.toolbar{{display:none!important}}.paper{{width:auto;max-width:none;margin:0;padding:0;box-shadow:none}}.ocr-notes{{display:none!important}}p{{orphans:3;widows:3}}a{{color:inherit;text-decoration:none}}}}\n\'\'\'\n    checked = " checked" if comparison else ""\n    klass = \' class="show-ocr"\' if comparison else ""\n    return (\'<!doctype html><html lang="vi"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\'\n            + \'<title>\' + html.escape(title) + \'</title><style>\' + css + \'</style></head><body\' + klass + \'>\'\n            + \'<div class="toolbar"><button onclick="window.print()">In / lưu PDF</button>\'\n            + \'<label><input type="checkbox"\' + checked + \' onchange="document.body.classList.toggle(\\\'show-ocr\\\',this.checked)"> Hiện OCR để đối chiếu trên màn hình</label>\'\n            + \'<small>Bản in dùng nội dung và ảnh gốc ở kích thước DOCX; phần OCR đối chiếu không được in kèm. Bố cục xuống dòng/ngắt trang HTML có thể khác Word. PDF giữ bố cục gốc được cung cấp riêng khi chuyển đổi thành công.</small></div>\'\n            + \'<main class="paper"><article class="paper-inner">\' + body + unplaced + ocr_notes + \'</article></main></body></html>\')\n\n\ndef original_layout_pdf(source, output_dir, allow_install=True):\n    """Best-effort original-DOCX render, explicitly not an OCR-derived PDF."""\n    source, output_dir = Path(source), Path(output_dir)\n    executable = shutil.which("soffice")\n    for attempt in range(2):\n        if executable:\n            try:\n                with tempfile.TemporaryDirectory(prefix="print-profile-", dir=output_dir.parent) as profile:\n                    process = subprocess.run(\n                        [executable, "-env:UserInstallation=" + Path(profile).resolve().as_uri(), "--headless", "--norestore",\n                         "--convert-to", "pdf:writer_pdf_Export", "--outdir", str(output_dir), str(source)],\n                        capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=180,\n                    )\n                produced = output_dir / (source.stem + ".pdf")\n                if produced.is_file() and produced.stat().st_size > 100:\n                    target = output_dir / "de-thi-goc.pdf"\n                    if produced != target:\n                        produced.rename(target)\n                    return dict(ok=True, path=str(target), kind="original_docx_layout_not_ocr")\n                detail = (process.stdout + "\\n" + process.stderr).strip()\n            except (OSError, subprocess.SubprocessError) as error:\n                detail = str(error)\n        else:\n            detail = "Chưa có LibreOffice Writer."\n        if attempt == 0 and allow_install and sys.platform.startswith("linux") and shutil.which("apt-get"):\n            print("Bản in gốc: đang cài LibreOffice Writer lần đầu…", flush=True)\n            try:\n                install_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive", APT_LISTCHANGES_FRONTEND="none")\n                apt = ["apt-get", "-o", "Dpkg::Use-Pty=0", "-o", "DPkg::Lock::Timeout=60",\n                       "-o", "Acquire::Retries=1", "-o", "Acquire::http::Timeout=30", "-o", "Acquire::https::Timeout=30"]\n                print("Bản in gốc: cập nhật danh sách gói (giới hạn 240s)…", flush=True)\n                subprocess.run([*apt, "update"], check=True, timeout=240, env=install_env, stdin=subprocess.DEVNULL)\n                print("Bản in gốc: tải / cài Writer (giới hạn 480s)…", flush=True)\n                subprocess.run([*apt, "install", "-y", "--no-install-recommends", "libreoffice-writer"],\n                               check=True, timeout=480, env=install_env, stdin=subprocess.DEVNULL)\n                executable = shutil.which("soffice")\n                continue\n            except (OSError, subprocess.SubprocessError) as error:\n                detail += " " + str(error)\n        break\n    return dict(ok=False, error=detail, kind="original_docx_layout_not_ocr")\n\n\ndef export_exam_pages(source, extraction, plan, output_dir, title="Đề thi", pandoc="pandoc", make_pdf=True):\n    output_dir = Path(output_dir)\n    setup = read_print_setup(source)\n    process = subprocess.run(\n        [pandoc, "-f", "docx", str(extraction["docx"]), "-t", "html5", "--mathml", "--wrap=none",\n         "--extract-media=" + str(output_dir / "print-media")],\n        capture_output=True, text=True, encoding="utf-8", timeout=180,\n    )\n    if process.returncode:\n        raise RuntimeError("Không tạo được HTML bản in: " + process.stderr)\n    fragments = _render_ocr_fragments(plan["records"], pandoc, output_dir)\n    parser = _ExamBody(process.stdout, plan["records"], output_dir, fragments)\n    parser.feed(process.stdout)\n    recovered = [record for record in plan["records"] if record["id"] not in parser.placed]\n    unplaced = ""\n    if recovered:\n        unplaced = \'<section class="recovered"><h2>Ảnh/công thức khôi phục — cần kiểm tra vị trí</h2>\'\n        unplaced += "".join(\'<p>\' + parser.image(record, (record.get("occurrences") or [{}])[0]) + \'</p>\' for record in recovered)\n        unplaced += \'</section>\'\n    body = "".join(parser.output)\n    ocr_notes = \'<section class="ocr-notes"><h2>Kết quả OCR để đối chiếu — không in kèm đề</h2>\'\n    for index, record in enumerate(plan["records"], 1):\n        if record.get("ocr_text"):\n            fragment = fragments.get(record["id"]) or \'<pre>\' + html.escape(record["ocr_text"]) + \'</pre>\'\n            ocr_notes += \'<div><h3>Ảnh/công thức \' + str(index) + \'</h3>\' + source_image_html(record, (record.get("occurrences") or [{}])[0]) + fragment + \'</div>\'\n    ocr_notes += \'</section>\'\n    for filename, comparison in (("de-thi.html", False), ("doi-chieu-ocr.html", True)):\n        (output_dir / filename).write_text(_page_html(title, body, setup, comparison, unplaced, ocr_notes), encoding="utf-8")\n    report = dict(html="de-thi.html", comparison_html="doi-chieu-ocr.html", page=setup,\n                  print_content="source_docx_images_and_native_math", ocr_annotations="screen_only",\n                  recovered_unplaced_images=len(recovered),\n                  limitations=["HTML reflow có thể khác xuống dòng/ngắt trang của Word.",\n                               "Ảnh thả nổi và nhiều section không được tái tạo vị trí tuyệt đối trong HTML.",\n                               "PDF gốc có thể thay phông nếu máy chuyển đổi thiếu phông Word."])\n    report["original_pdf"] = original_layout_pdf(source, output_dir) if make_pdf else dict(ok=False, error="Không yêu cầu chuyển PDF gốc trong lượt này.")\n    (output_dir / "print-report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")\n    return report\n\n\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\nparser = argparse.ArgumentParser()\nparser.add_argument("--input", required=True)\nparser.add_argument("--output", required=True)\nparser.add_argument("--title", default="Đề thi")\nargs = parser.parse_args()\nINPUT_PATH = Path(args.input).resolve()\nif not INPUT_PATH.is_file() or INPUT_PATH.suffix.lower() != ".docx":\n    raise ValueError("Khong phai file DOCX: " + str(INPUT_PATH))\nwith zipfile.ZipFile(INPUT_PATH) as archive:\n    if not {"[Content_Types].xml", "word/document.xml"}.issubset(archive.namelist()):\n        raise ValueError("File khong co cau truc Word DOCX hop le")\n\nimport numpy as np\nfrom PIL import Image, ImageOps\nimport torch\n\nif not torch.cuda.is_available():\n    raise RuntimeError("Tien trinh OCR khong thay GPU CUDA; kiem tra Accelerator.")\nfree_bytes, total_bytes = torch.cuda.mem_get_info(0)\nif free_bytes < 7_000_000_000:\n    raise RuntimeError("Code goc yeu cau it nhat 7 GB VRAM trong tren GPU 0.")\nprint("GPU:", torch.cuda.get_device_name(0), flush=True)\nprint("INPUT:", INPUT_PATH, flush=True)\nprint("MODEL: deepseek-ai/DeepSeek-OCR | dtype: bfloat16 | attention: eager", flush=True)\n\nif not shutil.which("pandoc"):\n    import pypandoc\n    pandoc = Path(pypandoc.get_pandoc_path())\n    os.environ["PATH"] = str(pandoc.parent) + os.pathsep + os.environ.get("PATH", "")\n\nWORK_DIR = Path(args.output).resolve()\nWORK_DIR.mkdir(parents=True, exist_ok=False)\nMEDIA_DIR = WORK_DIR / "media"\nPREP_DIR = WORK_DIR / "preprocessed"\nOCR_DIR = WORK_DIR / "ocr_imgs"\nFINAL_PATH = WORK_DIR / "final.md"\nfor directory in (MEDIA_DIR, PREP_DIR, OCR_DIR):\n    directory.mkdir()\n\n# ================================================================\n# HELPER FUNCTIONS\n# ================================================================\ndef img_hash(path):\n    try:\n        img = Image.open(path).convert("RGB")\n        return hashlib.md5(np.array(img).tobytes()).hexdigest()\n    except Exception:\n        h = hashlib.md5()\n        with open(path, "rb") as f:\n            while True:\n                chunk = f.read(65536)\n                if not chunk: break\n                h.update(chunk)\n        return h.hexdigest()\n\ndef analyze_image_for_ocr(img_path):\n    try:\n        img_rgb = Image.open(img_path).convert("RGB")\n        small   = img_rgb.resize((64, 64))\n        colors  = small.getcolors(64*64)\n        if (len(colors) if colors else 9999) > 400:\n            return "diagram"\n        img  = Image.open(img_path).convert("L")\n        arr  = np.array(img, dtype=np.float32)\n        h, w = arr.shape\n        dark_ratio = (arr < 128).mean()\n        dy = np.abs(np.diff(arr, axis=0)); dx = np.abs(np.diff(arr, axis=1))\n        edge_density = (dy > 30).mean() + (dx > 30).mean()\n        bh, bw = max(1, h//8), max(1, w//8)\n        block_vars = [arr[i:i+bh, j:j+bw].std()\n                      for i in range(0, h-bh, bh) for j in range(0, w-bw, bw)]\n        bvar_spread = np.std(block_vars) if block_vars else 0\n        is_text    = dark_ratio > 0.03 and dark_ratio < 0.6 and edge_density > 0.08\n        is_diagram = dark_ratio < 0.15 and edge_density < 0.12 and bvar_spread > 20\n        if is_diagram and not is_text: return "diagram"\n        elif is_text:                  return "text"\n        else:                          return "mixed"\n    except Exception:\n        return "mixed"\n\ndef is_tiny_label_image(img_path, max_side=130):\n    try:\n        w, h = Image.open(img_path).size\n        return max(w, h) <= max_side\n    except Exception:\n        return False\n\ndef preprocess_for_ocr(img_path, out_dir):\n    try:\n        img = Image.open(img_path).convert("RGB")\n        w, h = img.size\n        arr  = np.array(img.convert("L"))\n        mask = (arr < 240)\n        rows = np.where(mask.any(axis=1))[0]; cols = np.where(mask.any(axis=0))[0]\n        if len(rows) > 0 and len(cols) > 0:\n            m = 8\n            cropped = img.crop((max(0,cols[0]-m), max(0,rows[0]-m),\n                                 min(w,cols[-1]+m), min(h,rows[-1]+m)))\n            if cropped.size[0]*cropped.size[1] < w*h*0.85:\n                img = cropped; w, h = img.size\n        if max(w, h) > 1200:\n            s = 1200/max(w,h)\n            img = img.resize((int(w*s), int(h*s)), Image.LANCZOS)\n        if np.array(img.convert("L")).std() < 40:\n            img = ImageOps.autocontrast(img, cutoff=2)\n        out = out_dir / ("prep_" + img_path.stem + ".png")\n        img.save(str(out), format="PNG", optimize=False)\n        return out\n    except Exception:\n        return img_path\n\ndef extract_alt_texts_from_docx(docx_path):\n    import zipfile\n    try:\n        with zipfile.ZipFile(docx_path, \'r\') as z:\n            xml_content = z.read(\'word/document.xml\').decode(\'utf-8\', errors=\'ignore\')\n        return [m.strip() for m in re.findall(r\'<wp:docPr\\s+[^>]*\\bdescr="([^"]*)"\', xml_content)]\n    except Exception:\n        return []\n\ndef is_useful_alt_text(text, min_len=15):\n    if not text or len(text.strip()) < min_len: return False\n    for pat in [r\'^Picture\\s*\\d*$\', r\'^Image\\s*\\d*$\', r\'^Hình\\s*\\d*$\',\n                r\'^A (picture|screenshot|diagram) containing\']:\n        if re.match(pat, text.strip(), re.IGNORECASE): return False\n    return True\n_GARBAGE_PATTERNS = [\n    r\'"Line":\\s*\\{\', r\'"line":\\s*\\[\', r\'\\(-\\d+\\.\\d+,\\s*\\d+\\.\\d+\\)\\s*--\',\n    r\'Image ID:\\s*[A-Z0-9]+\', r\'"line_type"\', r\'"line_endpoint"\',\n]\n\ndef is_garbage_ocr(text):\n    if not text or not text.strip(): return True\n    tokens = text.strip().split()\n    if len(tokens) > 5 and all(t.isdigit() for t in tokens[:10]): return True\n    for pat in _GARBAGE_PATTERNS:\n        if re.search(pat, text): return True\n    if "Point" in text and text.count("| 10 |") > 5: return True\n    if text.startswith("<smiles>") and len(text) > 500: return True\n    if "Category" in text and "Value" in text: return True\n    if text.count("| 100 |") > 5: return True\n    lines = [l.strip() for l in text.split(\'\\n\') if \'|\' in l and \'---\' not in l and l.strip()]\n    if len(lines) > 8:\n        values = []\n        for line in lines[1:]:\n            cells = [c.strip() for c in line.split(\'|\') if c.strip()]\n            if cells: values.append(cells[-1])\n        if len(values) > 8:\n            mc = max(set(values), key=values.count)\n            if values.count(mc) / len(values) > 0.8: return True\n    tokens2 = re.split(r\'[\\s|<>/]+\', text)\n    symbol_re = re.compile(r\'^[&#$%^*`]+\\d{0,2}\\*{0,2}$\')\n    if len([t for t in tokens2 if t and len(t)<=4 and symbol_re.match(t)]) >= 4: return True\n    return False\n\ndef clean_ocr_table(text):\n    if "<tr>" not in text.lower(): return text\n    if text.count("None") > 5 or text.count("<td></td>") > 10: return ""\n    rows = re.findall(r\'<tr>(.*?)</tr>\', text, re.DOTALL)\n    md = []\n    for i, row in enumerate(rows):\n        cells = [c.strip() for c in re.findall(r\'<td>(.*?)</td>\', row, re.DOTALL)]\n        if not cells or not any(cells): continue\n        md.append("| " + " | ".join(cells) + " |")\n        if i == 0: md.append("|" + " --- |"*len(cells))\n    result = "\\n".join(md)\n    if result.count("Total") > 3 or result.count("| 10 |") > 5: return ""\n    return result\n\ndef clean_ocr(text):\n    if not text: return ""\n    s = text.strip()\n    if is_garbage_ocr(s): return ""\n    if "<tr>" in s.lower():\n        c = clean_ocr_table(s)\n        if not c or is_garbage_ocr(c): return ""\n        return c\n    return s\n\ndef format_ocr_inline(ocr_text):\n    # Preserve LaTeX line breaks, matrices, multiplication and short formulas.\n    return ocr_text.strip() if ocr_text else ""\n\ndef clean_pandoc_output(text):\n    def merge_img_attr_regex(t):\n        def _r(m):\n            return m.group(1) + re.sub(r\'\\s+\', \' \', m.group(2)).strip()\n        return re.sub(r\'(!\\[\\]\\([^)\\n]+\\))\\s*\\n*\\s*(\\{[^}]*\\})\', _r, t, flags=re.DOTALL)\n    text = merge_img_attr_regex(text)\n\n    def fix_bracket_wide(m):\n        nums = re.findall(r\'\\d+\', m.group(1))\n        return "[" + nums[0] + ";" + nums[1] + ")" if len(nums) >= 2 else m.group(0)\n    text = re.sub(r\'\\\\mathbf\\{[^}]*\\}(.*?)(?:\\\\right\\)|\\))\', fix_bracket_wide, text, flags=re.DOTALL)\n    text = re.sub(r\'\\[\\\\?\\s*(\\d+)\\s*;\\s*(\\d+)\\s*\\)\', r\'[\\1;\\2)\', text)\n    text = re.sub(r\'\\*\\*\\[([A-D])\\.\\*\\*\', r\'**\\1.**\', text)\n\n    def fix_matrix_block(m):\n        content = m.group(1)\n        def _rm(mm):\n            inner = mm.group(1).replace(r\'\\&\',\' \').replace(\'\\\\&\',\' \')\n            lines = [re.sub(r\'\\s*&\\s*.*$\',\'\',re.sub(r\'^\\s*&\\s*\',\'\',l.strip()))\n                     for l in re.split(r\'\\\\\\\\\', inner) if l.strip()]\n            return "\\\\begin{cases}" + "\\\\\\\\".join(l for l in lines if l) + "\\\\end{cases}"\n        content = re.sub(\n            r\'\\\\left\\s*\\\\?\\{?\\s*\\\\begin\\{(?:matrix|array)[^}]*\\}(.*?)\'\n            r\'\\\\end\\{(?:matrix|array)\\}\\s*\\\\right[.\\|\\\\]*\', _rm, content, flags=re.DOTALL)\n        content = re.sub(r\'\\s{2,}\', \' \', re.sub(r\'[ \\t]*\\n[ \\t]*\', \' \', content)).strip()\n        return "$" + content + "$"\n    text = re.sub(r\'\\$((?:(?!\\$).){1,800}?\\\\begin\\{(?:matrix|array)\\}(?:(?!\\$).)*?)\\$\',\n                  fix_matrix_block, text, flags=re.DOTALL)\n    text = re.sub(r\'\\\\\\-\', \'-\', text)\n    text = re.sub(r\'^\\*\\*(-{2,}[^*]+)\\*\\*$\', r\'\\1\', text, flags=re.MULTILINE)\n    text = re.sub(r\'\\*\\*\\\\\\[GT\\\\\\]\\*\\*\', \'> **[GT]**\', text)\n    text = re.sub(r\'\\*\\*\\\\\\[([A-Z]+)\\\\\\]\\*\\*\', r\'> **[\\1]**\', text)\n    text = re.sub(r\'^\\\\\\[GT\\\\\\]$\', \'> **[GT]**\', text, flags=re.MULTILINE)\n    text = re.sub(r\'^\\\\\\[/[\\u2227\\u039B]?\\\\\\]$\', \'\', text, flags=re.MULTILINE)\n    text = re.sub(r\'^\\\\\\[/\\\\\\]$\', \'\', text, flags=re.MULTILINE)\n    text = text.replace(r\'\\lbrack\',\'[\').replace(r\'\\rbrack\',\']\')\n\n    def fix_bu(m):\n        return "**" + re.sub(r\'\\[([^\\]]+)\\]\\{\\.underline\\}\', r\'\\1\', m.group(1)) + "**"\n    text = re.sub(r\'\\*\\*([^*]*\\[[^\\]]+\\]\\{\\.underline\\}[^*]*)\\*\\*\', fix_bu, text)\n    text = re.sub(r\'\\[([^\\]]+)\\]\\{\\.underline\\}\', r\'**\\1**\', text)\n    text = re.sub(r\'\\*{4,}([^*\\n]+)\\*{2,}\\.?\\*{0,2}\',\n                  lambda m: "**"+m.group(1).rstrip(\'*\')+".**", text)\n    text = re.sub(r\'\\*{4,}\', \'**\', text)\n    text = re.sub(r\'\\bậc\\s+b\\s+a\\b\', \'bậc ba\', text)\n    text = re.sub(r\'\\bhàm s\\s+ố\\b\', \'hàm số\', text)\n    text = re.sub(r\'\\bsố h\\s+ạng\\b\', \'số hạng\', text)\n    text = re.sub(r\'\\bcấp s\\s+ố\\b\', \'cấp số\', text)\n    text = re.sub(r\'\\bđường th\\s+ẳng\\b\', \'đường thẳng\', text)\n\n    def fix_im(m):\n        f = m.group(1)\n        f = re.sub(r\'\\\\left\\.\\s*\',\'\',f); f = re.sub(r\'\\\\right\\.\\s*\',\'\',f)\n        f = re.sub(r\'^\\\\\\s+\\[\',\'[\',f); f = re.sub(r\'^\\\\\\s+\\{\',\'{\',f); f = re.sub(r\'^\\\\\\s+\',\'\',f)\n        if len(re.findall(r\'\\\\right\',f)) > len(re.findall(r\'\\\\left\',f)):\n            f = re.sub(r\'\\\\right\\)\',\')\',f); f = re.sub(r\'\\\\right\\]\',\']\',f)\n            f = re.sub(r\'\\\\right\\}\',\'}\',f); f = re.sub(r\'\\\\right\\|\',\'|\',f)\n        f = re.sub(r\'\\\\mathbf\\{([^a-zA-Z\\}]{1,3})\\}\', r\'\\1\', f)\n        cc = [r\'\\\\frac\',r\'\\\\sqrt\',r\'\\\\sum\',r\'\\\\int\',r\'\\\\prod\',r\'\\\\lim\',r\'\\\\begin\',r\'\\\\over\']\n        def sd(mm):\n            ld,c,rd = mm.group(1),mm.group(2).strip(),mm.group(3)\n            if any(re.search(x,c) for x in cc): return mm.group(0)\n            if len(c) <= 40:\n                d={\'(\':\'()\',\'[\':\'[]\',\'{\':\'{}\'}; o,cc2=d.get(ld,(ld,rd))\n                return o+c+(cc2[1] if len(cc2)>1 else rd)\n            return mm.group(0)\n        f = re.sub(r\'\\\\left([\\(\\[\\{])(.*?)\\\\right([\\)\\]\\}])\', sd, f, flags=re.DOTALL)\n        f = re.sub(r\'\\\\left\\|(.*?)\\\\right\\|\', lambda mm: "|"+mm.group(1).strip()+"|", f, flags=re.DOTALL)\n        return "$" + re.sub(r\'\\s{2,}\',\' \',f.strip()) + "$"\n    text = re.sub(r\'\\$([^$\\n]{1,600})\\$\', fix_im, text)\n\n    text = re.sub(r\'\\\\text\\{([^}]+)\\}\',\n                  lambda m: m.group(0) if \' \' in m.group(1) and len(m.group(1))>10 else m.group(1), text)\n\n    text = re.sub(r\'([^\\n])[ \\t]*(!\\[\\]\\([^)]+\\)(?:\\{[^}]*\\})?)\', r\'\\1\\n\\n\\2\', text)\n    text = re.sub(r\'(!\\[\\]\\([^)]+\\)(?:\\{[^}]*\\})?)[ \\t]*([^\\n>])\', r\'\\1\\n\\n\\2\', text)\n    text = re.sub(r\'(!\\[\\]\\([^)]+\\)(?:\\{[^}]*\\})?)\\n(> \\*\\*\\[OCR\\]\\*\\*)\', r\'\\1\\n\\n\\2\', text)\n    text = re.sub(r\'\\*\\*A\\.\\*\\*[^\\n]*(?:\\n+\\*\\*[BCD]\\.\\*\\*[^\\n]*){1,3}\',\n                  lambda m: re.sub(r\'\\n{1,}(\\*\\*[BCD]\\.\\*\\*)\', r\' \\1\', m.group(0)), text)\n    text = re.sub(r\'([^\\n])\\n(\\*\\*A\\.\\*\\*[^\\n])\', r\'\\1\\n\\n\\2\', text)\n    text = re.sub(r\'([^\\n])\\n(\\*\\*Câu\\s+\\d+)\', r\'\\1\\n\\n\\2\', text)\n    text = merge_img_attr_regex(text)\n    text = text.replace(\'**\',\'\').replace(\'*\',\'\')\n    text = re.sub(r\'\\n{4,}\', \'\\n\\n\\n\', text)\n    text = re.sub(r\'[ \\t]+$\', \'\', text, flags=re.MULTILINE)\n    text = re.sub(r\'^\\s+$\', \'\', text, flags=re.MULTILINE)\n    return text.strip()\n\n# ================================================================\nprint("\\n" + "="*60)\nprint("  BUOC 1: Boc bang code (Pandoc)")\nprint("="*60)\n\nt_total = time.time()\nt0 = time.time()\n\nMD_PATH = WORK_DIR / "step1.md"\nextraction = mark_docx_images(INPUT_PATH, WORK_DIR)\nproc = subprocess.run([\n    "pandoc", "-f", "docx", str(extraction["docx"]),\n    "-t", "markdown+tex_math_dollars", "--wrap=none",\n    "--extract-media=" + str(MEDIA_DIR), "-o", str(MD_PATH),\n], capture_output=True, text=True)\n\nif proc.returncode != 0:\n    raise RuntimeError("Pandoc loi: " + proc.stderr)\n\nmd_text = MD_PATH.read_text(encoding="utf-8")\nimage_plan = prepare_image_plan(md_text, extraction, WORK_DIR)\nmd_text = image_plan["markup"]\nimg_refs = re.findall(r\'!\\[[^\\]]*\\]\\(([^)]+)\\)\', md_text)\nn_formulas = md_text.count("$") // 2\nref_to_hash = image_plan["ref_to_hash"]\nneed_ocr = image_plan["tasks"]\nn_skip_diagram = 0\nn_alt_hit = 0\nprint("Ảnh trong nguồn:", image_plan["summary"]["images_in_docx"],\n      "| vector:", image_plan["summary"]["vector_images"],\n      "| ảnh cần kiểm tra:", image_plan["summary"]["images_unprocessed"], flush=True)\nfor image_record in image_plan["records"]:\n    if image_record.get("error"):\n        print("[CẦN KIỂM TRA]", image_record["member"], ":", image_record["error"], flush=True)\n\nt_code = time.time() - t0\nprint("  " + str(n_formulas) + " cong thuc | " + str(len(need_ocr)) + " anh can OCR | "\n      + str(n_skip_diagram) + " diagram skip | " + str(n_alt_hit) + " alt-text | "\n      + str(round(t_code,1)) + "s")\n\n# ================================================================\nocr_cache = {}  # Alt-text is metadata, never a replacement for actual image OCR.\nt_load = 0\nn_success = 0\n\nif need_ocr:\n    print("\\n" + "="*60)\n    print("  BUOC 2: Load DeepSeek-OCR")\n    print("="*60)\n\n    from transformers import AutoModel, AutoTokenizer, StoppingCriteria, StoppingCriteriaList\n\n    class GarbageStoppingCriteria(StoppingCriteria):\n        def __init__(self, tokenizer, check_every=15, window=80, min_symbol_tokens=4):\n            self.tokenizer = tokenizer; self.check_every = check_every\n            self.window = window; self.min_symbol_tokens = min_symbol_tokens\n            self.step = 0\n            self._re = re.compile(r\'^[&#$%^*`]+\\d{0,2}\\*{0,2}$\')\n        def __call__(self, input_ids, scores, **kwargs):\n            self.step += 1\n            if self.step % self.check_every != 0: return False\n            if input_ids.shape[1] < self.window: return False\n            try:\n                text = self.tokenizer.decode(input_ids[0][-self.window:], skip_special_tokens=True)\n            except Exception:\n                return False\n            toks = re.split(r\'[\\s|<>/]+\', text)\n            return len([t for t in toks if t and len(t)<=4 and self._re.match(t)]) >= self.min_symbol_tokens\n\n    t0 = time.time()\n    gc.collect(); torch.cuda.empty_cache()\n    MODEL_NAME = "deepseek-ai/DeepSeek-OCR"\n    print("  Dang load " + MODEL_NAME + " (lan dau tai ~6.7GB)...")\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, use_fast=True)\n    model = AutoModel.from_pretrained(\n        MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16,\n        attn_implementation="eager", low_cpu_mem_usage=True).eval().cuda()\n\n    _orig = model.generate\n    _stopper = GarbageStoppingCriteria(tokenizer)\n    def _capped(self, *a, **kw):\n        _stopper.step = 0\n        kw["max_new_tokens"] = 1000\n        kw["eos_token_id"] = tokenizer.eos_token_id\n        kw["pad_token_id"] = tokenizer.eos_token_id or tokenizer.pad_token_id\n        if "stopping_criteria" not in kw or kw["stopping_criteria"] is None:\n            kw["stopping_criteria"] = StoppingCriteriaList([_stopper])\n        else:\n            kw["stopping_criteria"].append(_stopper)\n        return _orig(*a, **kw)\n    model.generate = types.MethodType(_capped, model)\n    t_load = time.time() - t0\n    print("  OK | VRAM: " + str(round(torch.cuda.memory_allocated()/1e9,1)) + "GB | " + str(round(t_load,1)) + "s")\n\n    print("\\n" + "="*60)\n    print("  BUOC 3: OCR " + str(len(need_ocr)) + " anh")\n    print("="*60)\n\n    def ocr_img(img_path, out_dir):\n        out_dir.mkdir(parents=True, exist_ok=True)\n        if is_tiny_label_image(img_path):\n            configs = [dict(crop_mode=False, base_size=384, image_size=384)]\n        else:\n            try:\n                w,h = Image.open(img_path).size\n                small = max(w,h) <= 350\n            except Exception:\n                small = False\n            configs = ([dict(crop_mode=False, base_size=448, image_size=448),\n                        dict(crop_mode=False, base_size=512, image_size=512),\n                        dict(crop_mode=False, base_size=640, image_size=640)] if small else\n                       [dict(crop_mode=False, base_size=512, image_size=512),\n                        dict(crop_mode=False, base_size=640, image_size=640),\n                        dict(crop_mode=False, base_size=448, image_size=448)])\n        for i, cfg in enumerate(configs):\n            try:\n                gc.collect(); torch.cuda.empty_cache()\n                with torch.inference_mode():\n                    model.infer(tokenizer, prompt="<image>\\nFree OCR.",\n                        image_file=str(img_path), output_path=str(out_dir/("a"+str(i))),\n                        save_results=True, test_compress=False, **cfg)\n                r = out_dir/("a"+str(i))/"result.mmd"\n                t = r.read_text(encoding="utf-8").strip() if r.exists() else ""\n                if not clean_ocr(t): continue\n                return t\n            except torch.cuda.OutOfMemoryError:\n                torch.cuda.empty_cache(); torch.cuda.synchronize()\n        return ""\n\n    t0_ocr = time.time()\n    for i, (ref, path, h) in enumerate(need_ocr):\n        t0 = time.time()\n        print("\\n  [" + str(i+1) + "/" + str(len(need_ocr)) + "] " + Path(ref).name)\n        raw_text = ocr_img(path, OCR_DIR/("img_"+str(i).zfill(3)))\n        text = clean_ocr(raw_text)\n        ocr_cache[h] = text\n        record_ocr_result(image_plan, h, raw_text, text)\n        write_image_report(image_plan, WORK_DIR)\n        dt = time.time()-t0\n        if text:\n            n_success += 1\n            print("       OK " + str(round(dt,1)) + "s | " + str(len(text)) + " ky tu")\n        else:\n            print("       [rac] " + str(round(dt,1)) + "s | bo qua")\n\n    t_ocr = time.time() - t0_ocr\n    print("\\n  OCR xong | " + str(round(t_ocr,1)) + "s | " + str(n_success) + "/" + str(len(need_ocr)) + " thanh cong")\n    try: del model\n    except: pass\n    gc.collect(); torch.cuda.empty_cache()\nelse:\n    print("\\n  Khong co anh can OCR - bo qua Buoc 2-3!")\n\n# ================================================================\nprint("\\n" + "="*60)\nprint("  BUOC 4: Ghep ket qua + Fix format")\nprint("="*60)\n\nfor ref in img_refs:\n    if ref not in ref_to_hash:\n        try: ref_to_hash[ref] = img_hash((MD_PATH.parent/ref).resolve())\n        except: pass\n\ndef build_final(md_text, ref_to_hash, ocr_cache):\n    def _s(m):\n        ref, width = m.group(1), m.group(2) or ""\n        h = ref_to_hash.get(ref, "")\n        raw = ocr_cache.get(h, "").strip() if h else ""\n        fmt = format_ocr_inline(raw)\n        tag = "![](" + ref + ")" + width\n        block = "\\n".join("> " + line if line else ">" for line in fmt.splitlines())\n        return tag if not fmt else tag + "\\n\\n> [OCR]\\n" + block + "\\n"\n    return re.sub(r\'!\\[[^\\]]*\\]\\(([^)]+)\\)(\\{[^}]*\\})?\', _s, md_text)\n\n# Do not run Markdown-emphasis cleanup over recognized formulas (e.g. x*y).\nfinal = build_final(clean_pandoc_output(md_text), ref_to_hash, ocr_cache)\nFINAL_PATH.write_text(final, encoding="utf-8")\nt_end = time.time()\n\nprint("\\n" + "="*60)\nprint("  HOAN THANH!")\nprint("="*60)\nprint("\\n  File ket qua : " + str(FINAL_PATH))\nprint("  Kich thuoc   : " + str(len(final)) + " ky tu | " + str(final.count(chr(10))) + " dong")\nprint("\\n  THOI GIAN:")\nprint("     Buoc 1 : " + str(round(t_code,1)) + "s")\nif need_ocr:\n    print("     Buoc 2 : " + str(round(t_load,1)) + "s (load model)")\n    print("     Buoc 3 : " + str(round(t_ocr,1)) + "s (OCR " + str(len(need_ocr)) + " anh)")\nprint("     TONG   : " + str(round(t_end-t_total,1)) + "s")\nprint("="*60)\n\nprint("\\n" + "="*60)\nprint("  NOI DUNG KET QUA")\nprint("="*60 + "\\n")\nprint(final)\n\n\n# Packaging/metrics only: inference and filtering above remain unchanged.\nportable = final.replace(str(WORK_DIR) + "/", "")\nFINAL_PATH.write_text(portable, encoding="utf-8")\nimage_summary = write_image_report(image_plan, WORK_DIR)\nmetrics = {\n    "pipeline": "attachment_pandoc_deepseek_ocr",\n    "model": "deepseek-ai/DeepSeek-OCR",\n    "dtype": "bfloat16",\n    "attention": "eager",\n    "native_formulas": n_formulas,\n    "images_referenced": len(img_refs),\n    "unique_ocr_images": len(need_ocr),\n    "ocr_successes": n_success,\n    "diagram_skips": n_skip_diagram,\n    "alt_text_hits": n_alt_hit,\n    "parse_preprocess_s": t_code,\n    "model_load_s": t_load,\n    "ocr_s": t_ocr if need_ocr else 0.0,\n    "pipeline_s": t_end - t_total,\n    "gpu_inference_ran": bool(need_ocr),\n    "image_policy": "all_nonblank_images",\n    "images_in_docx": image_summary["images_in_docx"],\n    "native_word_equations": image_summary["native_word_equations"],\n    "vector_images": image_summary["vector_images"],\n    "image_conversion_errors": image_summary["conversion_errors"],\n    "images_unprocessed": image_summary["images_unprocessed"],\n    "blank_images": image_summary["blank_images"],\n    "duplicate_images": image_summary["duplicate_images"],\n}\n(WORK_DIR / "metrics.json").write_text(\n    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"\n)\nprint_started = time.perf_counter()\ntry:\n    print("Đang tạo bản in theo kích thước ảnh trong DOCX…", flush=True)\n    print_report = export_exam_pages(INPUT_PATH, extraction, image_plan, WORK_DIR, title=args.title)\n    metrics["print_html"] = print_report["html"]\n    metrics["print_comparison_html"] = print_report["comparison_html"]\n    metrics["original_layout_pdf"] = print_report["original_pdf"].get("ok", False)\n    metrics["print_unplaced_images"] = print_report["recovered_unplaced_images"]\nexcept Exception as error:\n    metrics["print_export_error"] = str(error)\n    print("[BẢN IN CẦN KIỂM TRA]", error, flush=True)\nmetrics["print_export_s"] = time.perf_counter() - print_started\n(WORK_DIR / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")\nprint("RESULT:", FINAL_PATH, flush=True)\n\n'
APP_BUILD = '450b1b66bdc0'

# Isolated namespace: never reuse INPUT_FOLDER/choose_docx from an older cell.
app_namespace = {"__name__": "hybrid_clean_ui"}
exec(compile(APP_SOURCE, "<hybrid-clean-ui>", "exec"), app_namespace)
hybrid_app = app_namespace["launch"](
    globals().get("HYBRID_APP_ROOT", "/kaggle/working/hybrid_ocr_clean"),
    WORKER_SOURCE,
)
